# Day 28 — Linear Algebra with NumPy
 
**Week 5: NumPy — Numerical Computing Foundation**
 
Topics covered: `np.linalg` module, dot product, matrix multiplication (`@`), transpose, determinant, inverse, rank, trace, eigenvalues/eigenvectors, SVD, solving linear systems, norms, `np.linalg.lstsq`, Cholesky decomposition, QR decomposition.
 
---
 

### Q1. Dot Product — 1D Vectors
Create two 1D arrays `a = [1, 2, 3]` and `b = [4, 5, 6]`. Compute their dot product three ways: `np.dot(a, b)`, `a @ b`, and manually (element-wise multiply then sum). Verify all three give the same result.

In [1]:
# Q1: Dot product of 1D vectors — three ways

import numpy as np

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Method 1: np.dot()
dot_np = np.dot(a, b)

# Method 2: @ operator (matrix multiplication for 1D = dot product)
dot_at = a @ b

# Method 3: Manual — element-wise multiply then sum
dot_manual = np.sum(a * b)

print("np.dot(a, b)   :", dot_np)
print("a @ b          :", dot_at)
print("manual sum     :", dot_manual)
print("All equal?     :", dot_np == dot_at == dot_manual)

np.dot(a, b)   : 32
a @ b          : 32
manual sum     : 32
All equal?     : True


### How it works

- The **dot product** multiplies each pair of elements and adds the results: (1×4) + (2×5) + (3×6) = 32.
- `np.dot(a, b)` is the classic NumPy function for this.
- `a @ b` is the `@` operator — Python's dedicated matrix/dot product operator (added in Python 3.5).
- The manual way uses `a * b` (element-wise multiply) to get `[4, 10, 18]`, then `np.sum()` adds them up.
- All three methods produce the same result because they implement the same math.

**Key takeaway:** For 1D arrays, `np.dot`, `@`, and element-wise multiply-then-sum are three ways to compute the same dot product.

### Q2. Matrix Multiplication (`@` operator)
Create a `(2, 3)` matrix `A` and a `(3, 4)` matrix `B`. Compute `A @ B` and `np.matmul(A, B)`. Print the result shape. Explain why `np.dot` and `@` differ for 3D arrays (in a comment).

In [2]:
# Q2: Matrix multiplication with @ and np.matmul

import numpy as np

# (2, 3) matrix
A = np.array([[1, 2, 3],
              [4, 5, 6]])

# (3, 4) matrix
B = np.array([[1,  2,  3,  4],
              [5,  6,  7,  8],
              [9, 10, 11, 12]])

result_at     = A @ B           # @ operator
result_matmul = np.matmul(A, B) # np.matmul — identical for 2D

print("A @ B:\n", result_at)
print("\nnp.matmul(A, B):\n", result_matmul)
print("\nResult shape:", result_at.shape)  # (2, 4): rows of A × cols of B
print("Results equal?", np.array_equal(result_at, result_matmul))

# --- Why @ and np.dot differ for 3D arrays ---
# np.dot(A, B) for 3D: treats last axis of A and second-to-last axis of B
#   → can produce unexpected high-dimensional output
# np.matmul / @ for 3D: treats the arrays as a STACK of 2D matrices
#   and multiplies each pair — much more predictable for batched ops
# Rule of thumb: always use @ or np.matmul for matrix multiplication

A @ B:
 [[ 38  44  50  56]
 [ 83  98 113 128]]

np.matmul(A, B):
 [[ 38  44  50  56]
 [ 83  98 113 128]]

Result shape: (2, 4)
Results equal? True


### How it works

- Matrix multiplication is valid only when the **inner dimensions match**: `(2, 3) @ (3, 4)` works because both share `3`.
- The output shape is always `(rows of A, cols of B)` — here `(2, 4)`.
- `@` and `np.matmul` are identical for 2D arrays; use whichever reads more clearly.
- For 3D arrays, `np.dot` treats axes in a non-obvious way and can produce surprising shapes. `@` / `np.matmul` treats 3D as a **batch of matrices** (like a stack of pages) and multiplies each page pair — much easier to reason about.

**Key takeaway:** Always prefer `@` or `np.matmul` for matrix multiplication; reserve `np.dot` only for simple 1D dot products.

### Q3. Element-Wise vs Matrix Multiplication
Given two 3×3 matrices, show the difference between `A * B` (element-wise, Hadamard product) and `A @ B` (true matrix multiplication). Create a case where the results are dramatically different to illustrate the distinction.
 

In [3]:
# Q3: Element-wise (*) vs matrix multiplication (@) on 3x3 matrices

import numpy as np

A = np.array([[1, 2, 3],
              [0, 1, 0],
              [4, 0, 1]])

B = np.array([[5, 0, 0],
              [0, 5, 0],
              [0, 0, 5]])  # 5 × Identity — dramatic difference case

# Element-wise (Hadamard): each position multiplied independently
hadamard = A * B

# True matrix multiplication: rows × columns dot products
matmul = A @ B

print("A:\n", A)
print("\nB (5 × Identity):\n", B)
print("\nA * B  (element-wise / Hadamard):\n", hadamard)
print("\nA @ B  (matrix multiplication):\n", matmul)

# Show how different the results are
print("\nDifference (A@B  minus  A*B):\n", matmul - hadamard)
print("\nMax absolute difference:", np.max(np.abs(matmul - hadamard)))

A:
 [[1 2 3]
 [0 1 0]
 [4 0 1]]

B (5 × Identity):
 [[5 0 0]
 [0 5 0]
 [0 0 5]]

A * B  (element-wise / Hadamard):
 [[5 0 0]
 [0 5 0]
 [0 0 5]]

A @ B  (matrix multiplication):
 [[ 5 10 15]
 [ 0  5  0]
 [20  0  5]]

Difference (A@B  minus  A*B):
 [[ 0 10 15]
 [ 0  0  0]
 [20  0  0]]

Max absolute difference: 20


### How it works

- `A * B` (Hadamard product) multiplies **position by position**: `result[i,j] = A[i,j] × B[i,j]`. No rows or columns interact.
- `A @ B` (matrix multiplication) computes **dot products of rows and columns**: every output cell depends on an entire row of A and an entire column of B.
- The example uses `B = 5 × Identity` to make the contrast vivid:
  - `A * B` zeroes out every off-diagonal element of A (because Identity has zeros there).
  - `A @ B` simply scales all of A by 5 — a completely different outcome.
- Think of `*` as a per-pixel brightness adjustment on a photo, and `@` as rotating or transforming the entire image.

**Key takeaway:** `*` never mixes information between positions; `@` does — that mixing is exactly what makes matrix multiplication powerful in linear algebra and ML.

### Q4. Transpose
Create a 4×3 matrix. Compute its transpose using `.T`, `np.transpose()`, and `np.swapaxes(arr, 0, 1)`. Verify all give the same result. Then demonstrate that `(A @ B).T == B.T @ A.T` (the transpose of a product rule).

In [4]:
# Q4: Transpose three ways + verify the transpose-of-product rule

import numpy as np

A = np.array([[1,  2,  3],
              [4,  5,  6],
              [7,  8,  9],
              [10, 11, 12]])  # shape (4, 3)

# Three ways to transpose
t1 = A.T                      # attribute shorthand
t2 = np.transpose(A)          # function form
t3 = np.swapaxes(A, 0, 1)    # swap axis-0 (rows) and axis-1 (cols)

print("Original shape :", A.shape)       # (4, 3)
print("After .T       :", t1.shape)      # (3, 4)
print("All three equal?", np.array_equal(t1, t2) and np.array_equal(t2, t3))

# --- Transpose of a product rule: (A @ B).T == B.T @ A.T ---
# Need compatible shapes: use A (4,3) and B (3,2)
B = np.array([[1, 2],
              [3, 4],
              [5, 6]])  # shape (3, 2)

lhs = (A @ B).T        # transpose the product  → shape (2, 4)
rhs = B.T @ A.T        # multiply the transposes in REVERSED order → shape (2, 4)

print("\n(A @ B).T shape :", lhs.shape)
print("B.T @ A.T shape :", rhs.shape)
print("\n(A @ B).T:\n", lhs)
print("\nB.T @ A.T:\n", rhs)
print("\nRule holds?", np.allclose(lhs, rhs))

Original shape : (4, 3)
After .T       : (3, 4)
All three equal? True

(A @ B).T shape : (2, 4)
B.T @ A.T shape : (2, 4)

(A @ B).T:
 [[ 22  49  76 103]
 [ 28  64 100 136]]

B.T @ A.T:
 [[ 22  49  76 103]
 [ 28  64 100 136]]

Rule holds? True


### How it works

- Transposing **flips rows and columns**: a `(4, 3)` matrix becomes `(3, 4)`. Element `[i, j]` moves to `[j, i]`.
- `.T`, `np.transpose()`, and `np.swapaxes(arr, 0, 1)` all do the same thing for 2D arrays — three different spellings, one operation.
- The **transpose-of-product rule** says: `(A @ B).T == B.T @ A.T` — note the order is **reversed**.
- Think of it like putting on shoes and socks: to undo it, you take off shoes first, then socks — reversed order.
- `np.allclose` is used instead of `==` to safely handle any tiny floating-point rounding differences.

**Key takeaway:** Transposing a product reverses the order of the factors — `(A @ B).T = B.T @ A.T` — a rule that appears constantly in ML math (backpropagation, normal equations, etc.).

### Q5. Trace and Diagonal
Create a 4×4 matrix. Compute its trace using `np.trace()` and verify it equals `np.sum(np.diag(matrix))`. Use `np.fill_diagonal` to set diagonal elements to a specific value. Use `np.diag` with an offset `k` to extract super- and sub-diagonals.

In [5]:
# Q5: Trace, diagonal extraction, fill_diagonal, and offset diagonals

import numpy as np

M = np.array([[ 1,  2,  3,  4],
              [ 5,  6,  7,  8],
              [ 9, 10, 11, 12],
              [13, 14, 15, 16]])  # shape (4, 4)

# --- Trace: sum of main diagonal elements ---
trace_func = np.trace(M)                  # built-in
trace_manual = np.sum(np.diag(M))         # np.diag extracts main diagonal as 1D array

print("Matrix M:\n", M)
print("\nMain diagonal    :", np.diag(M))          # [1, 6, 11, 16]
print("np.trace(M)      :", trace_func)            # 34
print("sum(np.diag(M))  :", trace_manual)          # 34
print("Both equal?      :", trace_func == trace_manual)

# --- fill_diagonal: set main diagonal to a specific value ---
M_copy = M.copy()                         # preserve original
np.fill_diagonal(M_copy, 99)             # modifies in-place
print("\nAfter fill_diagonal(M, 99):\n", M_copy)

# --- Offset diagonals with k parameter ---
# k=+1 → one above main diagonal (super-diagonal)
super_diag = np.diag(M, k=1)
# k=-1 → one below main diagonal (sub-diagonal)
sub_diag   = np.diag(M, k=-1)
# k=+2 → two above main diagonal
super2_diag = np.diag(M, k=2)

print("\nSuper-diagonal (k=+1) :", super_diag)    # [2, 7, 12]
print("Sub-diagonal   (k=-1) :", sub_diag)        # [5, 10, 15]
print("Super-diagonal (k=+2) :", super2_diag)     # [3, 8]

Matrix M:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]

Main diagonal    : [ 1  6 11 16]
np.trace(M)      : 34
sum(np.diag(M))  : 34
Both equal?      : True

After fill_diagonal(M, 99):
 [[99  2  3  4]
 [ 5 99  7  8]
 [ 9 10 99 12]
 [13 14 15 99]]

Super-diagonal (k=+1) : [ 2  7 12]
Sub-diagonal   (k=-1) : [ 5 10 15]
Super-diagonal (k=+2) : [3 8]


### How it works

- The **trace** is the sum of the main diagonal elements (top-left to bottom-right). `np.trace()` computes it directly; `np.sum(np.diag(M))` does the same in two steps.
- `np.diag(M)` on a 2D matrix **extracts** the diagonal as a 1D array. On a 1D array, it does the reverse — builds a 2D matrix with that array on the diagonal.
- `np.fill_diagonal` sets diagonal values **in-place** (modifies the original), so always `.copy()` first if you need to keep the original.
- The `k` offset shifts which diagonal you target:
  - `k=0` → main diagonal (default)
  - `k=+1` → one step above (shorter by 1 element)
  - `k=-1` → one step below (shorter by 1 element)

**Key takeaway:** `np.diag` and `np.trace` are your main tools for diagonal operations; use `k` offsets to reach diagonals above or below the main one.

### Q6. Determinant
Compute the determinant of a 3×3 matrix using `np.linalg.det`. Create: a singular matrix (determinant = 0), an identity matrix (determinant = 1), and a 4×4 random matrix. Explain what a determinant of 0 means for the matrix's invertibility.
 

In [6]:
# Q6: Determinant — singular, identity, and random 4x4 matrices

import numpy as np

# --- Regular 3x3 matrix ---
A = np.array([[2, 1, 3],
              [0, 4, 1],
              [5, 2, 8]], dtype=float)

det_A = np.linalg.det(A)
print("Matrix A:\n", A)
print("det(A):", round(det_A, 6))

# --- Singular matrix: one row is a linear combination of others ---
# Row 3 = Row 1 + Row 2  → matrix is "flat", no unique solution exists
S = np.array([[1, 2, 3],
              [4, 5, 6],
              [5, 7, 9]], dtype=float)   # row3 = row1 + row2

det_S = np.linalg.det(S)
print("\nSingular matrix S (row3 = row1 + row2):\n", S)
print("det(S):", round(det_S, 6))       # 0 (or ~0 due to float rounding)
print("Is singular (|det| < 1e-10)?", abs(det_S) < 1e-10)

# --- Identity matrix: determinant always = 1 ---
I = np.eye(3)
det_I = np.linalg.det(I)
print("\nIdentity matrix I:\n", I)
print("det(I):", round(det_I, 6))       # exactly 1.0

# --- Random 4x4 matrix ---
np.random.seed(42)
R = np.random.randint(1, 10, size=(4, 4)).astype(float)
det_R = np.linalg.det(R)
print("\nRandom 4x4 matrix R:\n", R)
print("det(R):", round(det_R, 6))

# --- Invertibility summary ---
print("\n--- Invertibility check ---")
for name, det in [("A", det_A), ("S (singular)", det_S),
                  ("I (identity)", det_I), ("R (random 4x4)", det_R)]:
    invertible = abs(det) > 1e-10
    print(f"{name:20s} | det = {det:10.4f} | invertible: {invertible}")

Matrix A:
 [[2. 1. 3.]
 [0. 4. 1.]
 [5. 2. 8.]]
det(A): 5.0

Singular matrix S (row3 = row1 + row2):
 [[1. 2. 3.]
 [4. 5. 6.]
 [5. 7. 9.]]
det(S): 0.0
Is singular (|det| < 1e-10)? True

Identity matrix I:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
det(I): 1.0

Random 4x4 matrix R:
 [[7. 4. 8. 5.]
 [7. 3. 7. 8.]
 [5. 4. 8. 8.]
 [3. 6. 5. 2.]]
det(R): -251.0

--- Invertibility check ---
A                    | det =     5.0000 | invertible: True
S (singular)         | det =     0.0000 | invertible: False
I (identity)         | det =     1.0000 | invertible: True
R (random 4x4)       | det =  -251.0000 | invertible: True


### How it works

- The **determinant** is a single number that summarises key properties of a square matrix.
- `det = 0` means the matrix is **singular**: its rows (or columns) are not independent — one row can be built from the others. A singular matrix cannot be inverted and the system `Ax = b` has no unique solution.
- `det = 1` for the identity matrix because it represents "do nothing" — no scaling or rotation.
- A non-zero determinant means the matrix **can be inverted** and the linear system it represents has exactly one solution.
- Think of the determinant as the **scaling factor** of the transformation: `det = 0` means the transformation collapses space into a lower dimension (a 3D cube becomes a flat sheet), so there is no way to reverse it.

**Key takeaway:** `det = 0` → singular → not invertible → no unique solution; any non-zero determinant means the matrix is invertible.

### Q7. Matrix Inverse
Compute the inverse of a 3×3 invertible matrix using `np.linalg.inv`. Verify: `A @ inv(A) ≈ I` (identity matrix, approximately due to floating point). Use `np.allclose` to check. Show what happens with a singular matrix.

In [7]:
# Q7: Matrix inverse, verification, and singular matrix behaviour

import numpy as np

# --- Invertible 3x3 matrix ---
A = np.array([[2, 1, 3],
              [0, 4, 1],
              [5, 2, 8]], dtype=float)

A_inv = np.linalg.inv(A)
print("Matrix A:\n", A)
print("\nInverse of A:\n", A_inv.round(6))

# --- Verify: A @ inv(A) should equal Identity ---
product = A @ A_inv
print("\nA @ inv(A):\n", product.round(10))  # tiny floats visible without rounding
print("\nEquals Identity? (np.allclose):", np.allclose(product, np.eye(3)))

# Also verify the other direction: inv(A) @ A ≈ I
print("inv(A) @ A ≈ I?             :", np.allclose(A_inv @ A, np.eye(3)))

# --- Show floating-point imprecision directly ---
print("\nRaw diagonal (should be 1.0):", np.diag(product))
print("Raw off-diagonal sample     :", product[0, 1])  # near 0 but not exact

# --- Singular matrix: inv() raises LinAlgError ---
S = np.array([[1, 2, 3],
              [4, 5, 6],
              [5, 7, 9]], dtype=float)  # row3 = row1 + row2, det = 0

print("\nSingular matrix S:\n", S)
print("det(S):", round(np.linalg.det(S), 10))

try:
    S_inv = np.linalg.inv(S)
    print("Inverse:\n", S_inv)
except np.linalg.LinAlgError as e:
    print(f"LinAlgError caught: {e}")   # Singular matrix

Matrix A:
 [[2. 1. 3.]
 [0. 4. 1.]
 [5. 2. 8.]]

Inverse of A:
 [[ 6.  -0.4 -2.2]
 [ 1.   0.2 -0.4]
 [-4.   0.2  1.6]]

A @ inv(A):
 [[ 1.  0. -0.]
 [ 0.  1.  0.]
 [ 0.  0.  1.]]

Equals Identity? (np.allclose): True
inv(A) @ A ≈ I?             : True

Raw diagonal (should be 1.0): [1. 1. 1.]
Raw off-diagonal sample     : 0.0

Singular matrix S:
 [[1. 2. 3.]
 [4. 5. 6.]
 [5. 7. 9.]]
det(S): 0.0
Inverse:
 [[ 1.12589991e+15  1.12589991e+15 -1.12589991e+15]
 [-2.25179981e+15 -2.25179981e+15  2.25179981e+15]
 [ 1.12589991e+15  1.12589991e+15 -1.12589991e+15]]


### How it works

- The **inverse** of matrix A (written A⁻¹) is the matrix that "undoes" A: `A @ A⁻¹ = I` (identity).
- Think of it like division: if A multiplies a vector, A⁻¹ brings it back — just as multiplying by 3 then by 1/3 returns the original number.
- `np.allclose` is used instead of `==` because floating-point arithmetic introduces tiny rounding errors (e.g. `1.0000000000000002` instead of `1.0`). `np.allclose` treats values within a small tolerance as equal.
- Inverting a **singular matrix** (det = 0) raises `np.linalg.LinAlgError` because no inverse exists — the transformation collapsed space and cannot be reversed.
- Always wrap `np.linalg.inv` in a `try/except` when the matrix comes from real data, as near-singular matrices can cause numerical instability.

**Key takeaway:** `A @ inv(A) ≈ I` always holds for invertible matrices; use `np.allclose` to verify because exact equality fails due to floating-point rounding.

### Q8. Solving Linear Systems — `np.linalg.solve`
Set up and solve the system of linear equations:
```
2x + y - z = 8
-3x - y + 2z = -11
-2x + y + 2z = -3
```
Express as `Ax = b`, then solve with `np.linalg.solve`. Verify by computing `A @ solution` and comparing to `b`.

In [8]:
# Q8: Solve a system of linear equations with np.linalg.solve

import numpy as np

# --- Express the system as Ax = b ---
# 2x +  y -  z =  8
#-3x -  y + 2z = -11
#-2x +  y + 2z = -3

# Each row of A = coefficients of [x, y, z] in one equation
A = np.array([[ 2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]], dtype=float)

# b = right-hand side values
b = np.array([8, -11, -3], dtype=float)

print("Coefficient matrix A:\n", A)
print("\nRight-hand side b:", b)

# --- Solve: finds x such that A @ x = b ---
solution = np.linalg.solve(A, b)
x, y, z = solution
print("\nSolution:")
print(f"  x = {x:.6f}")
print(f"  y = {y:.6f}")
print(f"  z = {z:.6f}")

# --- Verify: A @ solution should equal b ---
b_reconstructed = A @ solution
print("\nVerification — A @ solution:", b_reconstructed)
print("Original b              :", b)
print("Match (np.allclose)?    :", np.allclose(b_reconstructed, b))

# --- Manual substitution check ---
print("\nManual check:")
print(f"  2x + y - z  = {2*x + y - z:.1f}  (expected  8)")
print(f" -3x - y + 2z = {-3*x - y + 2*z:.1f} (expected -11)")
print(f" -2x + y + 2z = {-2*x + y + 2*z:.1f}  (expected -3)")

Coefficient matrix A:
 [[ 2.  1. -1.]
 [-3. -1.  2.]
 [-2.  1.  2.]]

Right-hand side b: [  8. -11.  -3.]

Solution:
  x = 2.000000
  y = 3.000000
  z = -1.000000

Verification — A @ solution: [  8. -11.  -3.]
Original b              : [  8. -11.  -3.]
Match (np.allclose)?    : True

Manual check:
  2x + y - z  = 8.0  (expected  8)
 -3x - y + 2z = -11.0 (expected -11)
 -2x + y + 2z = -3.0  (expected -3)


### How it works

- Any system of linear equations can be written as `Ax = b`: A holds the coefficients, x holds the unknowns, b holds the results.
- `np.linalg.solve(A, b)` finds x directly — it is faster and more numerically stable than computing `inv(A) @ b`, which is the naive approach.
- Think of it as solving `3x = 9` → `x = 3`, but generalised to multiple equations and unknowns simultaneously.
- Verification `A @ solution ≈ b` confirms correctness: plug the answer back in and check you recover the right-hand side.
- `np.linalg.solve` requires A to be **square and invertible** (non-singular); use `np.linalg.lstsq` instead when the system is over- or under-determined.

**Key takeaway:** Prefer `np.linalg.solve(A, b)` over `inv(A) @ b` — same answer, but faster and numerically safer.

### Q9. `np.linalg.lstsq` — Least Squares Solution
Given an overdetermined system (more equations than unknowns — simulate with 5 noisy data points on a line), use `np.linalg.lstsq` to find the best-fit line coefficients. Plot the points and fitted line (describe what you'd plot, or compute predicted values). Compare the residuals.

In [9]:
# Q9: Least squares best-fit line through noisy data points

import numpy as np

# --- Simulate 5 noisy points roughly following y = 2x + 3 ---
np.random.seed(0)
x_vals = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
noise  = np.random.randn(5) * 0.5          # small random noise
y_vals = 2 * x_vals + 3 + noise            # true line + noise

print("x values :", x_vals)
print("y values :", y_vals.round(4))

# --- Build matrix A for system: y = m*x + c ---
# Each row: [x_i, 1]  →  unknowns are [m, c]
# Shape (5, 2): 5 equations, 2 unknowns → overdetermined (no exact solution)
A = np.column_stack([x_vals, np.ones(5)])
print("\nDesign matrix A (shape {}):".format(A.shape))
print(A)

# --- Solve with lstsq: minimises ||Ax - b||² ---
# rcond=None silences a deprecation warning
coeffs, residuals, rank, sv = np.linalg.lstsq(A, y_vals, rcond=None)
m, c = coeffs
print(f"\nBest-fit line: y = {m:.4f}x + {c:.4f}")
print(f"True line was: y = 2.0000x + 3.0000")

# --- Predicted values and per-point residuals ---
y_pred    = A @ coeffs                     # predicted y for each x
point_res = y_vals - y_pred                # how far each point is from the line

print("\nPoint-by-point comparison:")
print(f"{'x':>4} {'y_actual':>10} {'y_pred':>10} {'residual':>10}")
for xi, ya, yp, r in zip(x_vals, y_vals, y_pred, point_res):
    print(f"{xi:>4.1f} {ya:>10.4f} {yp:>10.4f} {r:>10.4f}")

# --- Total residual (sum of squared errors) ---
sse = np.sum(point_res ** 2)
print(f"\nSum of squared residuals (SSE): {sse:.6f}")
print(f"lstsq returned residuals array: {residuals.round(6)}")  # same value
print(f"Match? {np.allclose(sse, residuals)}")

# --- What the plot would show ---
print("\n--- Plot description ---")
print("Scatter: 5 noisy (x, y) points")
print(f"Line   : y = {m:.4f}x + {c:.4f}  drawn from x=1 to x=5")
print("Residuals: vertical dashed lines from each point to the fitted line")

x values : [1. 2. 3. 4. 5.]
y values : [ 5.882   7.2001  9.4894 12.1204 13.9338]

Design matrix A (shape (5, 2)):
[[1. 1.]
 [2. 1.]
 [3. 1.]
 [4. 1.]
 [5. 1.]]

Best-fit line: y = 2.1024x + 3.4180
True line was: y = 2.0000x + 3.0000

Point-by-point comparison:
   x   y_actual     y_pred   residual
 1.0     5.8820     5.5204     0.3617
 2.0     7.2001     7.6228    -0.4227
 3.0     9.4894     9.7251    -0.2358
 4.0    12.1204    11.8275     0.2929
 5.0    13.9338    13.9299     0.0039

Sum of squared residuals (SSE): 0.450857
lstsq returned residuals array: [0.450857]
Match? True

--- Plot description ---
Scatter: 5 noisy (x, y) points
Line   : y = 2.1024x + 3.4180  drawn from x=1 to x=5
Residuals: vertical dashed lines from each point to the fitted line


### How it works

- An **overdetermined system** has more equations than unknowns (5 equations, 2 unknowns here) — no exact solution exists, so we find the **best approximate solution** instead.
- `np.linalg.lstsq` minimises the **sum of squared residuals** (SSE): it finds the line that makes the total squared vertical distance from points to line as small as possible.
- The **design matrix A** has shape `(5, 2)`: each row is `[x, 1]` so the unknowns `[m, c]` map to slope and intercept.
- `lstsq` returns four values: coefficients, residuals (total SSE), matrix rank, and singular values — usually only the first two are needed.
- A **residual** for each point = actual y − predicted y; small residuals mean the line fits well.

**Key takeaway:** Use `np.linalg.lstsq` when you have more equations than unknowns — it finds the line (or plane) that gets as close as possible to all points at once, minimising total squared error.

### Q10. Matrix Rank
Use `np.linalg.matrix_rank` to compute the rank of: a full-rank 3×3 matrix, a singular 3×3 matrix, a 4×3 matrix, and a zero matrix. Explain what rank means and how it relates to linear independence of rows/columns.

In [10]:
# Q10: Matrix rank — full-rank, singular, rectangular, and zero matrix

import numpy as np

# --- 1. Full-rank 3x3 matrix: all rows linearly independent ---
A_full = np.array([[1, 0, 0],
                   [0, 2, 0],
                   [0, 0, 3]], dtype=float)  # diagonal → obviously independent

rank_full = np.linalg.matrix_rank(A_full)
print("Full-rank 3x3 matrix:\n", A_full)
print(f"Rank: {rank_full}  (max possible = min(3,3) = 3) → full rank\n")

# --- 2. Singular 3x3 matrix: one row is a combo of others ---
A_sing = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [5, 7, 9]], dtype=float)  # row3 = row1 + row2

rank_sing = np.linalg.matrix_rank(A_sing)
print("Singular 3x3 matrix (row3 = row1 + row2):\n", A_sing)
print(f"Rank: {rank_sing}  (only 2 independent rows, rank < 3 → singular)\n")

# --- 3. Rectangular 4x3 matrix ---
A_rect = np.array([[1, 2, 3],
                   [0, 1, 4],
                   [5, 6, 0],
                   [2, 0, 7]], dtype=float)  # 4 rows, 3 cols

rank_rect = np.linalg.matrix_rank(A_rect)
print("Rectangular 4x3 matrix:\n", A_rect)
print(f"Rank: {rank_rect}  (max possible = min(4,3) = 3) → full column rank\n")

# --- 4. Zero matrix: no independent rows at all ---
A_zero = np.zeros((3, 3))

rank_zero = np.linalg.matrix_rank(A_zero)
print("Zero matrix:\n", A_zero)
print(f"Rank: {rank_zero}  (no information at all → rank 0)\n")

# --- Summary table ---
print("--- Summary ---")
print(f"{'Matrix':<25} {'Shape':<10} {'Rank':<6} {'Max possible rank'}")
print(f"{'Full-rank 3x3':<25} {'(3,3)':<10} {rank_full:<6} {min(3,3)}")
print(f"{'Singular 3x3':<25} {'(3,3)':<10} {rank_sing:<6} {min(3,3)}")
print(f"{'Rectangular 4x3':<25} {'(4,3)':<10} {rank_rect:<6} {min(4,3)}")
print(f"{'Zero 3x3':<25} {'(3,3)':<10} {rank_zero:<6} {min(3,3)}")

# --- Rank vs determinant connection ---
print("\n--- Rank and determinant connection (square matrices only) ---")
print(f"det(A_full): {np.linalg.det(A_full):.4f}  → non-zero → full rank → invertible")
print(f"det(A_sing): {np.linalg.det(A_sing):.4f}   → zero     → rank < 3  → singular")

Full-rank 3x3 matrix:
 [[1. 0. 0.]
 [0. 2. 0.]
 [0. 0. 3.]]
Rank: 3  (max possible = min(3,3) = 3) → full rank

Singular 3x3 matrix (row3 = row1 + row2):
 [[1. 2. 3.]
 [4. 5. 6.]
 [5. 7. 9.]]
Rank: 2  (only 2 independent rows, rank < 3 → singular)

Rectangular 4x3 matrix:
 [[1. 2. 3.]
 [0. 1. 4.]
 [5. 6. 0.]
 [2. 0. 7.]]
Rank: 3  (max possible = min(4,3) = 3) → full column rank

Zero matrix:
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Rank: 0  (no information at all → rank 0)

--- Summary ---
Matrix                    Shape      Rank   Max possible rank
Full-rank 3x3             (3,3)      3      3
Singular 3x3              (3,3)      2      3
Rectangular 4x3           (4,3)      3      3
Zero 3x3                  (3,3)      0      3

--- Rank and determinant connection (square matrices only) ---
det(A_full): 6.0000  → non-zero → full rank → invertible
det(A_sing): 0.0000   → zero     → rank < 3  → singular


### How it works

- **Rank** = the number of **linearly independent** rows (or columns) in a matrix. Independent means no row can be built by adding/scaling other rows.
- Think of rank as "how much unique information the matrix carries": a rank-2 matrix inside a 3×3 grid means one row adds nothing new.
- For an `(m, n)` matrix, **maximum possible rank = min(m, n)** — you can never have more independent rows than the smaller dimension.
- Key rank facts:
  - Full-rank square matrix → invertible, det ≠ 0, unique solution exists
  - Rank < min(m, n) → called **rank-deficient**; rows/columns are redundant
  - Zero matrix → rank 0, carries no information at all
- `np.linalg.matrix_rank` uses SVD internally and counts singular values above a small tolerance — robust against floating-point noise.

**Key takeaway:** Rank tells you how many truly independent directions a matrix spans; a rank-deficient matrix has redundant rows and cannot be inverted.

### Q11. Norms
Compute various norms of a vector `[3, -4, 0, 5, -2]` using `np.linalg.norm`:
- L1 norm (Manhattan distance, `ord=1`)
- L2 norm (Euclidean length, `ord=2`, default)
- L-infinity norm (max absolute value, `ord=np.inf`)
- Frobenius norm of a 3×3 matrix
Verify the L2 norm manually (square root of sum of squares).

### Q12. Unit Vector (Normalization)
Write a function `normalize(v)` that returns the unit vector in the same direction as `v` using `np.linalg.norm`. Handle the zero vector case. Then apply it to normalize each row of a 2D matrix (producing a matrix of unit-length row vectors).

In [11]:
# Q11: Vector and matrix norms

import numpy as np

v = np.array([3, -4, 0, 5, -2], dtype=float)
print("Vector v:", v)

# --- L1 norm: sum of absolute values (Manhattan distance) ---
l1 = np.linalg.norm(v, ord=1)
l1_manual = np.sum(np.abs(v))
print(f"\nL1 norm  (ord=1) : {l1:.4f}  | manual |3|+|-4|+|0|+|5|+|-2| = {l1_manual:.4f}")

# --- L2 norm: square root of sum of squares (Euclidean length) ---
l2 = np.linalg.norm(v, ord=2)          # ord=2 is the default
l2_manual = np.sqrt(np.sum(v ** 2))    # manual verification
print(f"L2 norm  (ord=2) : {l2:.4f}  | manual sqrt(9+16+0+25+4) = {l2_manual:.4f}")
print(f"L2 verified?     : {np.isclose(l2, l2_manual)}")

# --- L-infinity norm: largest absolute value in the vector ---
linf = np.linalg.norm(v, ord=np.inf)
linf_manual = np.max(np.abs(v))
print(f"L∞ norm  (ord=∞) : {linf:.4f}  | manual max(|v|) = {linf_manual:.4f}")

# --- Frobenius norm of a 3x3 matrix ---
# Frobenius = L2 norm "flattened": sqrt of sum of all squared elements
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)

frob      = np.linalg.norm(M)              # default for matrices = Frobenius
frob_ord  = np.linalg.norm(M, ord='fro')  # explicit Frobenius
frob_manual = np.sqrt(np.sum(M ** 2))

print(f"\nMatrix M:\n{M}")
print(f"\nFrobenius norm (default)  : {frob:.6f}")
print(f"Frobenius norm (ord='fro'): {frob_ord:.6f}")
print(f"Manual sqrt(sum(M²))      : {frob_manual:.6f}")
print(f"All equal?                : {np.allclose([frob, frob_ord], frob_manual)}")

# --- Summary comparison ---
print("\n--- Norm summary for v = [3, -4, 0, 5, -2] ---")
print(f"L1  (sum of |elements|)       : {l1:.4f}")
print(f"L2  (Euclidean length)        : {l2:.4f}")
print(f"L∞  (largest |element|)       : {linf:.4f}")
print(f"\nNote: L1 >= L2 >= L∞ always holds for the same vector")
print(f"Check: {l1:.4f} >= {l2:.4f} >= {linf:.4f} → {l1 >= l2 >= linf}")

Vector v: [ 3. -4.  0.  5. -2.]

L1 norm  (ord=1) : 14.0000  | manual |3|+|-4|+|0|+|5|+|-2| = 14.0000
L2 norm  (ord=2) : 7.3485  | manual sqrt(9+16+0+25+4) = 7.3485
L2 verified?     : True
L∞ norm  (ord=∞) : 5.0000  | manual max(|v|) = 5.0000

Matrix M:
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]

Frobenius norm (default)  : 16.881943
Frobenius norm (ord='fro'): 16.881943
Manual sqrt(sum(M²))      : 16.881943
All equal?                : True

--- Norm summary for v = [3, -4, 0, 5, -2] ---
L1  (sum of |elements|)       : 14.0000
L2  (Euclidean length)        : 7.3485
L∞  (largest |element|)       : 5.0000

Note: L1 >= L2 >= L∞ always holds for the same vector
Check: 14.0000 >= 7.3485 >= 5.0000 → True


### How it works

- A **norm** measures the "size" or "length" of a vector — different norms measure it differently.
- **L1 (Manhattan):** sum of absolute values — like walking city blocks, only horizontal/vertical moves allowed.
- **L2 (Euclidean):** straight-line distance from origin — the familiar "length" of a vector using Pythagoras. This is the default when no `ord` is given.
- **L∞ (infinity):** just the largest absolute value — as the `ord` grows, smaller elements become irrelevant and only the biggest survives.
- **Frobenius norm:** the L2 norm applied to a matrix as if it were one long flattened vector — square root of all squared elements.
- The ordering `L1 ≥ L2 ≥ L∞` always holds: L1 adds everything up, L2 shrinks it via squaring, L∞ keeps only the max.

**Key takeaway:** L1, L2, and L∞ are the three most common norms; L2 is the default Euclidean length, while L1 and L∞ appear frequently in ML regularisation (Lasso uses L1, robustness checks use L∞).

### Q13. Eigenvalues and Eigenvectors
Given a 3×3 symmetric matrix, compute eigenvalues and eigenvectors using `np.linalg.eig`. Verify the fundamental property: `A @ v == λ * v` for each eigenvalue λ and corresponding eigenvector v. Use `np.linalg.eigh` for symmetric matrices and explain why it's preferred.

In [12]:
# Q13: Eigenvalues and eigenvectors — eig vs eigh, and verification

import numpy as np

# Symmetric matrix (A = A.T) — common in ML (covariance, PCA, etc.)
A = np.array([[4, 2, 0],
              [2, 3, 1],
              [0, 1, 2]], dtype=float)

print("Symmetric matrix A:\n", A)
print("Is symmetric?", np.allclose(A, A.T))

# --- np.linalg.eig: general eigensolver (works on any square matrix) ---
eigenvalues_eig,  eigenvectors_eig  = np.linalg.eig(A)

# --- np.linalg.eigh: specialised for symmetric/Hermitian matrices ---
# Returns eigenvalues in ASCENDING order (eig order is arbitrary)
eigenvalues_eigh, eigenvectors_eigh = np.linalg.eigh(A)

print("\n--- eig  eigenvalues  :", eigenvalues_eig.round(6))
print("--- eigh eigenvalues  :", eigenvalues_eigh.round(6))  # sorted ascending
print("\nAll real? (eigh guarantees real):", np.all(np.isreal(eigenvalues_eigh)))

# --- Verify: A @ v == λ * v for each eigenpair (using eigh results) ---
print("\n--- Verification: A @ v  vs  λ * v ---")
for i in range(len(eigenvalues_eigh)):
    lam = eigenvalues_eigh[i]
    v   = eigenvectors_eigh[:, i]   # column i = eigenvector i
    lhs = A @ v                     # matrix × eigenvector
    rhs = lam * v                   # scalar × eigenvector
    match = np.allclose(lhs, rhs)
    print(f"  λ{i+1} = {lam:.4f} | A@v = {lhs.round(4)} | λv = {rhs.round(4)} | match: {match}")

# --- Why eigh is preferred for symmetric matrices ---
print("\n--- eig  eigenvectors (columns):\n", eigenvectors_eig.round(6))
print("\n--- eigh eigenvectors (columns):\n", eigenvectors_eigh.round(6))

# eigh eigenvectors are guaranteed orthonormal: V.T @ V = I
VtV = eigenvectors_eigh.T @ eigenvectors_eigh
print("\neigh: V.T @ V = I ?", np.allclose(VtV, np.eye(3)))

# eig does NOT guarantee sorted or orthonormal output for symmetric matrices
print("eig : V.T @ V = I ?", np.allclose(eigenvectors_eig.T @ eigenvectors_eig, np.eye(3)))

# --- Reconstruct A from eigendecomposition: A = V @ diag(λ) @ V.T ---
V   = eigenvectors_eigh
lam = np.diag(eigenvalues_eigh)
A_reconstructed = V @ lam @ V.T
print("\nA reconstructed from V @ Λ @ V.T:\n", A_reconstructed.round(6))
print("Matches original A?", np.allclose(A_reconstructed, A))

Symmetric matrix A:
 [[4. 2. 0.]
 [2. 3. 1.]
 [0. 1. 2.]]
Is symmetric? True

--- eig  eigenvalues  : [5.669079 2.476024 0.854897]
--- eigh eigenvalues  : [0.854897 2.476024 5.669079]

All real? (eigh guarantees real): True

--- Verification: A @ v  vs  λ * v ---
  λ1 = 0.8549 | A@v = [ 0.3693 -0.5807  0.5072] | λv = [ 0.3693 -0.5807  0.5072] | match: True
  λ2 = 2.4760 | A@v = [ 1.2165 -0.9269 -1.9472] | λv = [ 1.2165 -0.9269 -1.9472] | match: True
  λ3 = 5.6691 | A@v = [-4.2876 -3.5782 -0.9752] | λv = [-4.2876 -3.5782 -0.9752] | match: True

--- eig  eigenvectors (columns):
 [[-0.75632   0.491296  0.431981]
 [-0.631179 -0.374362 -0.679313]
 [-0.172027 -0.786436  0.593233]]

--- eigh eigenvectors (columns):
 [[ 0.431981  0.491296 -0.75632 ]
 [-0.679313 -0.374362 -0.631179]
 [ 0.593233 -0.786436 -0.172027]]

eigh: V.T @ V = I ? True
eig : V.T @ V = I ? True

A reconstructed from V @ Λ @ V.T:
 [[ 4.  2. -0.]
 [ 2.  3.  1.]
 [ 0.  1.  2.]]
Matches original A? True


### How it works

- An **eigenvector** v of matrix A is a special direction that A only stretches or shrinks — never rotates. The **eigenvalue** λ is the stretch factor: `A @ v = λ * v`.
- Think of it like a rubber sheet: most arrows change direction when you pull it, but eigenvectors only change in length (by factor λ).
- `np.linalg.eig` works on any square matrix but returns eigenvalues in arbitrary order and eigenvectors that may not be orthogonal.
- `np.linalg.eigh` is designed for **symmetric matrices** and gives three guarantees `eig` does not: eigenvalues are always **real**, always **sorted ascending**, and eigenvectors are always **orthonormal** (`V.T @ V = I`).
- A symmetric matrix can always be perfectly **reconstructed** as `V @ Λ @ V.T` — this is the eigendecomposition, the foundation of PCA and spectral methods.

**Key takeaway:** Always use `np.linalg.eigh` (not `eig`) for symmetric matrices — it is faster, more numerically stable, and guarantees real sorted eigenvalues with orthonormal eigenvectors.

### Q14. Eigenvalues — Real-World Intuition
Create a 2×2 transformation matrix `[[2, 1], [0, 3]]`. Compute its eigenvalues and eigenvectors. Explain what the eigenvalues tell you about how the matrix stretches space (use specific numerical examples — what does it do to vectors along the eigenvector directions?).

In [13]:
# Q14: Eigenvalue intuition — how a matrix stretches space

import numpy as np

A = np.array([[2, 1],
              [0, 3]], dtype=float)

eigenvalues, eigenvectors = np.linalg.eig(A)

print("Transformation matrix A:\n", A)
print("\nEigenvalues  λ:", eigenvalues)
print("Eigenvectors V (columns):\n", eigenvectors.round(6))

# --- Extract individual eigenpairs ---
lam1, lam2 = eigenvalues
v1 = eigenvectors[:, 0]   # eigenvector for λ1
v2 = eigenvectors[:, 1]   # eigenvector for λ2

print(f"\nEigenpair 1:  λ = {lam1:.1f},  v = {v1.round(4)}")
print(f"Eigenpair 2:  λ = {lam2:.1f},  v = {v2.round(4)}")

# --- Verify stretch: A @ v == λ * v ---
print("\n--- Stretch verification ---")
for lam, v, label in [(lam1, v1, "λ1"), (lam2, v2, "λ2")]:
    stretched = A @ v
    scaled    = lam * v
    print(f"  {label}: A @ v = {stretched.round(4)}  |  λ*v = {scaled.round(4)}  |  match: {np.allclose(stretched, scaled)}")

# --- Concrete intuition: scale the eigenvectors and transform ---
print("\n--- What happens to vectors ALONG eigenvector directions ---")
for lam, v, label in [(lam1, v1, "v1 (λ=2)"), (lam2, v2, "v2 (λ=3)")]:
    for scale in [1.0, 2.0, 5.0]:
        input_vec  = scale * v
        output_vec = A @ input_vec
        actual_scale = np.linalg.norm(output_vec) / np.linalg.norm(input_vec)
        print(f"  {label} | input={input_vec.round(3)} → output={output_vec.round(3)} | stretch={actual_scale:.1f}x")

# --- What happens to a NON-eigenvector (direction changes!) ---
print("\n--- What happens to a vector NOT along an eigenvector direction ---")
u = np.array([1.0, 1.0])          # arbitrary vector
u_transformed = A @ u
angle_before = np.degrees(np.arctan2(u[1], u[0]))
angle_after  = np.degrees(np.arctan2(u_transformed[1], u_transformed[0]))
print(f"  Input  [1, 1]      : angle = {angle_before:.1f}°")
print(f"  Output A @ [1, 1]  : {u_transformed}  angle = {angle_after:.2f}°")
print(f"  Direction changed by {abs(angle_after - angle_before):.2f}° — NOT an eigenvector")

# --- Summary ---
print("\n--- Eigenvalue summary ---")
print(f"  λ1 = {lam1:.1f} → vectors along v1 are stretched by 2x, direction preserved")
print(f"  λ2 = {lam2:.1f} → vectors along v2 are stretched by 3x, direction preserved")
print(f"  All other vectors: BOTH direction and length change")

Transformation matrix A:
 [[2. 1.]
 [0. 3.]]

Eigenvalues  λ: [2. 3.]
Eigenvectors V (columns):
 [[1.       0.707107]
 [0.       0.707107]]

Eigenpair 1:  λ = 2.0,  v = [1. 0.]
Eigenpair 2:  λ = 3.0,  v = [0.7071 0.7071]

--- Stretch verification ---
  λ1: A @ v = [2. 0.]  |  λ*v = [2. 0.]  |  match: True
  λ2: A @ v = [2.1213 2.1213]  |  λ*v = [2.1213 2.1213]  |  match: True

--- What happens to vectors ALONG eigenvector directions ---
  v1 (λ=2) | input=[1. 0.] → output=[2. 0.] | stretch=2.0x
  v1 (λ=2) | input=[2. 0.] → output=[4. 0.] | stretch=2.0x
  v1 (λ=2) | input=[5. 0.] → output=[10.  0.] | stretch=2.0x
  v2 (λ=3) | input=[0.707 0.707] → output=[2.121 2.121] | stretch=3.0x
  v2 (λ=3) | input=[1.414 1.414] → output=[4.243 4.243] | stretch=3.0x
  v2 (λ=3) | input=[3.536 3.536] → output=[10.607 10.607] | stretch=3.0x

--- What happens to a vector NOT along an eigenvector direction ---
  Input  [1, 1]      : angle = 45.0°
  Output A @ [1, 1]  : [3. 3.]  angle = 45.00°
  Direction 

### How it works

- A matrix **transforms** vectors — it can rotate, stretch, or squish them. Most vectors change both direction and length after transformation.
- **Eigenvectors** are the special directions that only get **stretched or shrunk** — never rotated. The matrix leaves them pointing the same way.
- The **eigenvalue** is the **stretch factor** along that eigenvector direction:
  - λ = 2 → every vector along v1 doubles in length
  - λ = 3 → every vector along v2 triples in length
  - λ = 1 → no change; λ < 0 → flips direction; |λ| < 1 → shrinks
- Non-eigenvectors get pulled toward the **dominant eigenvector** (largest |λ|) with each repeated application of A — this is why eigenvalues matter in PCA, PageRank, and Markov chains.
- Think of it like a hill: eigenvector directions are the slope axes; eigenvalues tell you how steep each slope is.

**Key takeaway:** Eigenvalues measure how much a matrix stretches space along each eigenvector direction — only vectors exactly along those directions keep their orientation after transformation.

### Q15. Singular Value Decomposition (SVD)
Decompose a 4×3 matrix `A` using `np.linalg.svd`. This gives `U`, `S`, `Vt`. Reconstruct the original matrix from the components: `U @ np.diag(S) @ Vt`. Verify with `np.allclose`. Explain what each component represents.

In [14]:
# Q15: Singular Value Decomposition — decompose, reconstruct, verify

import numpy as np

A = np.array([[ 1,  2,  3],
              [ 4,  5,  6],
              [ 7,  8,  9],
              [10, 11, 12]], dtype=float)   # shape (4, 3)

print("Original matrix A (shape {}):\n".format(A.shape), A)

# --- Compute SVD ---
# U  : (4, 4) — left singular vectors  (orthonormal columns)
# S  : (3,)   — singular values        (always non-negative, descending order)
# Vt : (3, 3) — right singular vectors transposed (orthonormal rows)
U, S, Vt = np.linalg.svd(A)

print("\n--- SVD Components ---")
print(f"U  shape: {U.shape}  (left singular vectors)")
print(U.round(4))
print(f"\nS  shape: {S.shape}  (singular values, descending)")
print(S.round(4))
print(f"\nVt shape: {Vt.shape}  (right singular vectors, transposed)")
print(Vt.round(4))

# --- Reconstruct A = U @ diag(S) @ Vt ---
# np.diag(S) pads S into a (3,3) square diagonal matrix
# but U is (4,4), so we need a (4,3) diagonal — use full_matrices=False OR slice U
S_matrix = np.diag(S)                         # shape (3, 3)
U_reduced = U[:, :3]                          # take only first 3 cols of U → (4, 3)

A_reconstructed = U_reduced @ S_matrix @ Vt  # (4,3) @ (3,3) @ (3,3) = (4,3)

print("\n--- Reconstruction: U_reduced @ diag(S) @ Vt ---")
print("Reconstructed A:\n", A_reconstructed.round(6))
print("\nMatches original? (np.allclose):", np.allclose(A, A_reconstructed))

# --- Alternative: use full_matrices=False (economy/thin SVD) ---
U2, S2, Vt2 = np.linalg.svd(A, full_matrices=False)
A_rec2 = U2 @ np.diag(S2) @ Vt2
print("\nThin SVD shapes — U: {}, S: {}, Vt: {}".format(U2.shape, S2.shape, Vt2.shape))
print("Thin SVD reconstruction matches?", np.allclose(A, A_rec2))

# --- Verify U and Vt are orthonormal ---
print("\n--- Orthonormality checks ---")
print("U.T @ U = I ?", np.allclose(U.T @ U, np.eye(4)))   # full U
print("Vt @ Vt.T = I ?", np.allclose(Vt @ Vt.T, np.eye(3)))

# --- Singular values and matrix rank ---
tol = 1e-10
rank = np.sum(S > tol)
print(f"\nSingular values: {S.round(6)}")
print(f"Non-zero singular values: {rank}  → rank of A = {rank}")
print("(Matches np.linalg.matrix_rank?", np.linalg.matrix_rank(A) == rank, ")")

# --- Low-rank approximation: keep only top-1 singular value ---
k = 1
A_approx = (U_reduced[:, :k] * S[:k]) @ Vt[:k, :]
error = np.linalg.norm(A - A_approx, ord='fro')
print(f"\nRank-{k} approximation of A:\n", A_approx.round(4))
print(f"Frobenius error vs original: {error:.4f}")

Original matrix A (shape (4, 3)):
 [[ 1.  2.  3.]
 [ 4.  5.  6.]
 [ 7.  8.  9.]
 [10. 11. 12.]]

--- SVD Components ---
U  shape: (4, 4)  (left singular vectors)
[[-0.1409 -0.8247  0.54   -0.0917]
 [-0.3439 -0.4263 -0.6517  0.5247]
 [-0.547  -0.0278 -0.3167 -0.7744]
 [-0.7501  0.3706  0.4283  0.3414]]

S  shape: (3,)  (singular values, descending)
[25.4624  1.2907  0.    ]

Vt shape: (3, 3)  (right singular vectors, transposed)
[[-0.5045 -0.5745 -0.6445]
 [ 0.7608  0.0571 -0.6465]
 [-0.4082  0.8165 -0.4082]]

--- Reconstruction: U_reduced @ diag(S) @ Vt ---
Reconstructed A:
 [[ 1.  2.  3.]
 [ 4.  5.  6.]
 [ 7.  8.  9.]
 [10. 11. 12.]]

Matches original? (np.allclose): True

Thin SVD shapes — U: (4, 3), S: (3,), Vt: (3, 3)
Thin SVD reconstruction matches? True

--- Orthonormality checks ---
U.T @ U = I ? True
Vt @ Vt.T = I ? True

Singular values: [25.462407  1.290662  0.      ]
Non-zero singular values: 2  → rank of A = 2
(Matches np.linalg.matrix_rank? True )

Rank-1 approximation of 

### How it works

- SVD splits **any** matrix A into three parts: `A = U @ diag(S) @ Vt`
  - **U** — "output directions": orthonormal columns describing how A rotates/reflects the output space
  - **S** — "stretch amounts": singular values (always ≥ 0, sorted descending) — how much A scales along each direction
  - **Vt** — "input directions": orthonormal rows describing the input space orientations
- Think of it as three steps: Vt **rotates** the input, S **stretches** each axis, U **rotates** the result into output space.
- `full_matrices=False` (thin SVD) trims U to shape `(m, k)` where k = min(m, n) — avoids the padded columns that contribute nothing to reconstruction.
- The number of **non-zero singular values** equals the **rank** of the matrix.
- Keeping only the top-k singular values gives the best **rank-k approximation** of A — the foundation of image compression and PCA.

**Key takeaway:** SVD decomposes any matrix into rotation → stretch → rotation; singular values measure the "importance" of each dimension, making it the most powerful and general matrix decomposition in linear algebra.

### Q16. Low-Rank Approximation via SVD
Using a 6×5 matrix (or a small "image-like" matrix), compute its SVD. Reconstruct the matrix using only the top 1, 2, and 3 singular values (truncated SVD). Compute and print the reconstruction error (`np.linalg.norm(A - A_approx, 'fro')`) for each level of approximation.

In [15]:
# Q16: Low-rank approximation via truncated SVD at k=1, 2, 3

import numpy as np

# "Image-like" 6x5 matrix with some structure (not random — makes rank pattern clear)
A = np.array([[ 9,  4,  6,  8,  3],
              [ 2,  7,  1,  5,  9],
              [ 6,  1,  8,  3,  7],
              [ 4,  8,  2,  9,  1],
              [ 7,  3,  5,  6,  4],
              [ 1,  6,  9,  2,  8]], dtype=float)   # shape (6, 5)

print("Original matrix A (shape {}):\n".format(A.shape), A)

# --- Thin SVD: U (6,5), S (5,), Vt (5,5) ---
U, S, Vt = np.linalg.svd(A, full_matrices=False)

print(f"\nAll singular values: {S.round(4)}")
print(f"Total Frobenius norm of A: {np.linalg.norm(A, 'fro'):.4f}")

# --- Truncated SVD reconstruction for k = 1, 2, 3 ---
print("\n--- Truncated SVD Approximations ---")
print(f"{'k':<5} {'Singular values used':<30} {'Frob error':<14} {'% variance captured'}")
print("-" * 70)

total_variance = np.sum(S ** 2)          # total "energy" in matrix

for k in [1, 2, 3]:
    # Keep only top-k components: U[:, :k] is (6,k), diag(S[:k]) is (k,k), Vt[:k,:] is (k,5)
    A_approx = (U[:, :k] * S[:k]) @ Vt[:k, :]          # shape (6, 5)

    error        = np.linalg.norm(A - A_approx, 'fro')  # Frobenius reconstruction error
    var_captured = np.sum(S[:k] ** 2) / total_variance * 100

    print(f"k={k}   {str(S[:k].round(3)):<30} {error:<14.4f} {var_captured:.2f}%")

# --- Print each approximation matrix ---
print("\n--- Approximation matrices ---")
for k in [1, 2, 3]:
    A_approx = (U[:, :k] * S[:k]) @ Vt[:k, :]
    error    = np.linalg.norm(A - A_approx, 'fro')
    print(f"\nk={k}  (Frobenius error = {error:.4f}):\n", A_approx.round(2))

# --- Eckart-Young theorem: rank-k is the BEST possible rank-k approximation ---
# The error of rank-k truncation = sqrt(sum of squared unused singular values)
print("\n--- Eckart-Young theorem verification ---")
for k in [1, 2, 3]:
    A_approx      = (U[:, :k] * S[:k]) @ Vt[:k, :]
    frob_error    = np.linalg.norm(A - A_approx, 'fro')
    theorem_error = np.sqrt(np.sum(S[k:] ** 2))    # sqrt(σ_{k+1}² + ... + σ_r²)
    print(f"k={k}: actual error = {frob_error:.6f} | theorem predicts = {theorem_error:.6f} | match: {np.isclose(frob_error, theorem_error)}")

# --- Full reconstruction (k = min(m,n) = 5) as sanity check ---
A_full = (U * S) @ Vt
print(f"\nFull reconstruction (k=5) error: {np.linalg.norm(A - A_full, 'fro'):.2e}")
print("Matches original?", np.allclose(A, A_full))

Original matrix A (shape (6, 5)):
 [[9. 4. 6. 8. 3.]
 [2. 7. 1. 5. 9.]
 [6. 1. 8. 3. 7.]
 [4. 8. 2. 9. 1.]
 [7. 3. 5. 6. 4.]
 [1. 6. 9. 2. 8.]]

All singular values: [28.2436 10.5176  9.0648  4.6171  0.4344]
Total Frobenius norm of A: 31.8119

--- Truncated SVD Approximations ---
k     Singular values used           Frob error     % variance captured
----------------------------------------------------------------------
k=1   [28.244]                       14.6389        78.82%
k=2   [28.244 10.518]                10.1822        89.76%
k=3   [28.244 10.518  9.065]         4.6375         97.87%

--- Approximation matrices ---

k=1  (Frobenius error = 14.6389):
 [[5.78 5.55 6.13 6.44 6.14]
 [4.6  4.41 4.88 5.12 4.88]
 [4.85 4.65 5.14 5.39 5.14]
 [4.6  4.41 4.87 5.12 4.88]
 [4.83 4.63 5.12 5.37 5.12]
 [4.99 4.79 5.29 5.56 5.3 ]]

k=2  (Frobenius error = 10.1822):
 [[6.95 6.11 4.56 8.49 3.95]
 [4.19 4.21 5.44 4.39 5.67]
 [3.59 4.04 6.84 3.18 7.5 ]
 [6.57 5.37 2.2  8.6  1.17]
 [5.32 4.87 4.

### How it works

- **Truncated SVD** keeps only the top-k singular values and their corresponding vectors, discarding the rest. The result is the best possible rank-k approximation of A.
- Each singular value σ carries a portion of the matrix's total "energy" (variance). Larger σ = more important direction. Dropping small σ values loses little information.
- Reconstruction formula for rank-k: `A_approx = U[:, :k] @ diag(S[:k]) @ Vt[:k, :]` — shapes are `(m,k) @ (k,k) @ (k,n) = (m,n)`.
- **Frobenius error** measures total reconstruction loss across all elements — it decreases as k increases, reaching zero at full rank.
- The **Eckart-Young theorem** proves this is optimal: no other rank-k matrix can have a smaller Frobenius error than the truncated SVD. The exact error equals `sqrt(σ_{k+1}² + ... + σ_r²)`.
- In image compression, k=1 uses one "layer", k=2 adds a second refinement layer — each layer recovers more detail.

**Key takeaway:** Truncated SVD at rank-k is the mathematically optimal low-rank approximation; each additional singular value recovers more of the original matrix, with the Frobenius error dropping by exactly the energy of the dropped singular values.

### Q17. QR Decomposition
Decompose a 4×3 matrix `A` using `np.linalg.qr`. Verify: Q is orthogonal (`Q.T @ Q ≈ I`) and `Q @ R ≈ A`. Explain a use case for QR decomposition (e.g., solving least-squares problems, Gram-Schmidt orthogonalization).

In [16]:
# Q17: QR decomposition — verify properties and demonstrate use case

import numpy as np

A = np.array([[ 1,  2,  3],
              [ 4,  5,  6],
              [ 7,  8, 10],
              [ 2,  3,  5]], dtype=float)   # shape (4, 3), full column rank

print("Original matrix A (shape {}):\n".format(A.shape), A)

# --- Compute QR decomposition ---
# Q : (4, 3) — orthonormal columns (thin/economy mode by default)
# R : (3, 3) — upper triangular matrix
Q, R = np.linalg.qr(A)

print(f"\nQ shape: {Q.shape}  (orthonormal columns)")
print(Q.round(6))
print(f"\nR shape: {R.shape}  (upper triangular)")
print(R.round(6))

# --- Verify 1: Q is orthogonal → Q.T @ Q = I ---
QtQ = Q.T @ Q
print("\n--- Verification 1: Q.T @ Q = I ---")
print("Q.T @ Q:\n", QtQ.round(10))
print("Q is orthogonal?", np.allclose(QtQ, np.eye(3)))

# --- Verify 2: Q @ R reconstructs A ---
A_reconstructed = Q @ R
print("\n--- Verification 2: Q @ R = A ---")
print("Q @ R:\n", A_reconstructed.round(6))
print("Matches A?", np.allclose(A_reconstructed, A))

# --- R is upper triangular: all elements below diagonal are zero ---
print("\n--- Verify R is upper triangular ---")
lower_triangle = R[np.tril_indices(3, k=-1)]   # elements below main diagonal
print("Elements below diagonal in R:", lower_triangle.round(10))
print("All zero?", np.allclose(lower_triangle, 0))

# --- Use case: solving least-squares via QR (more stable than normal equations) ---
# Normal equations: x = inv(A.T @ A) @ A.T @ b  → can be numerically unstable
# QR approach: A = QR → Rx = Q.T @ b → back-substitution (stable)
print("\n--- Use case: Least-squares via QR vs np.linalg.lstsq ---")
b = np.array([1.0, 2.0, 3.0, 4.0])            # shape (4,) — overdetermined system

# Method 1: QR-based least squares
# A @ x = b  →  QR @ x = b  →  R @ x = Q.T @ b
QtB    = Q.T @ b                               # shape (3,)
x_qr   = np.linalg.solve(R, QtB)              # back-substitution via R

# Method 2: np.linalg.lstsq (also uses SVD internally)
x_lstsq, _, _, _ = np.linalg.lstsq(A, b, rcond=None)

print("Solution via QR   :", x_qr.round(6))
print("Solution via lstsq:", x_lstsq.round(6))
print("Both match?", np.allclose(x_qr, x_lstsq))

# --- Gram-Schmidt intuition: Q columns are orthonormal basis ---
print("\n--- Gram-Schmidt intuition ---")
print("Column norms of Q (should all be 1.0):")
for i in range(Q.shape[1]):
    print(f"  ||Q[:,{i}]|| = {np.linalg.norm(Q[:, i]):.6f}")
print("Dot products between columns (should all be ~0):")
for i in range(Q.shape[1]):
    for j in range(i+1, Q.shape[1]):
        print(f"  Q[:,{i}] · Q[:,{j}] = {np.dot(Q[:,i], Q[:,j]):.2e}")

Original matrix A (shape (4, 3)):
 [[ 1.  2.  3.]
 [ 4.  5.  6.]
 [ 7.  8. 10.]
 [ 2.  3.  5.]]

Q shape: (4, 3)  (orthonormal columns)
[[-0.119523  0.730297  0.291091]
 [-0.478091  0.182574  0.608645]
 [-0.83666  -0.365148 -0.18524 ]
 [-0.239046  0.547723 -0.714496]]

R shape: (3, 3)  (upper triangular)
[[ -8.3666   -10.03992  -12.788946]
 [  0.         1.095445   2.373464]
 [  0.         0.        -0.899735]]

--- Verification 1: Q.T @ Q = I ---
Q.T @ Q:
 [[ 1.  0.  0.]
 [ 0.  1. -0.]
 [ 0. -0.  1.]]
Q is orthogonal? True

--- Verification 2: Q @ R = A ---
Q @ R:
 [[ 1.  2.  3.]
 [ 4.  5.  6.]
 [ 7.  8. 10.]
 [ 2.  3.  5.]]
Matches A? True

--- Verify R is upper triangular ---
Elements below diagonal in R: [0. 0. 0.]
All zero? True

--- Use case: Least-squares via QR vs np.linalg.lstsq ---
Solution via QR   : [ 0.411765 -2.588235  2.117647]
Solution via lstsq: [ 0.411765 -2.588235  2.117647]
Both match? True

--- Gram-Schmidt intuition ---
Column norms of Q (should all be 1.0):
  ||Q

### How it works

- QR decomposition splits A into two parts: `A = Q @ R`
  - **Q** — orthonormal columns: every column has length 1 and all columns are perpendicular to each other (`Q.T @ Q = I`)
  - **R** — upper triangular: all zeros below the diagonal, making systems easy to solve by back-substitution
- Think of Q as a clean set of perpendicular "compass directions" that span the same space as A's columns — this is exactly what **Gram-Schmidt orthogonalisation** produces.
- **Why QR for least-squares?** The normal equation `inv(A.T @ A) @ A.T @ b` squares the condition number of A, amplifying numerical errors. QR avoids squaring — `R @ x = Q.T @ b` is solved by simple back-substitution, far more stable.
- `np.linalg.qr` returns the **thin/economy** Q by default: shape `(m, k)` where k = number of columns, not the full `(m, m)` padded version.
- QR also underpins the **QR algorithm** for computing eigenvalues iteratively — one of the most important algorithms in numerical linear algebra.

**Key takeaway:** QR gives you an orthonormal basis Q and an upper-triangular R; it is preferred over normal equations for least-squares because it is numerically stable and avoids squaring the condition number of A.


### Q18. Cholesky Decomposition
Create a symmetric positive definite matrix (construct it as `A = B.T @ B` for a random `B`). Compute its Cholesky decomposition `L = np.linalg.cholesky(A)`. Verify `L @ L.T ≈ A`. Explain when Cholesky is more efficient than LU decomposition.

In [17]:
# Q18: Cholesky decomposition — construct SPD matrix, decompose, verify

import numpy as np

np.random.seed(7)

# --- Construct a Symmetric Positive Definite (SPD) matrix ---
# Any matrix of form B.T @ B is guaranteed SPD (as long as B has full column rank)
B = np.random.randint(1, 6, size=(4, 4)).astype(float)
A = B.T @ B                               # shape (4, 4), guaranteed SPD

print("B:\n", B)
print("\nA = B.T @ B  (SPD matrix):\n", A)

# --- Verify A is symmetric and positive definite ---
is_symmetric = np.allclose(A, A.T)
eigenvalues  = np.linalg.eigh(A)[0]      # eigh guaranteed real for symmetric
is_pos_def   = np.all(eigenvalues > 0)   # all eigenvalues > 0 → positive definite

print(f"\nIs symmetric?        {is_symmetric}")
print(f"Eigenvalues:         {eigenvalues.round(4)}")
print(f"Is positive definite (all λ > 0)? {is_pos_def}")

# --- Cholesky decomposition: A = L @ L.T ---
# L is lower triangular with positive diagonal entries
L = np.linalg.cholesky(A)

print(f"\nL (lower triangular), shape {L.shape}:\n", L.round(6))

# --- Verify L is lower triangular: all elements above diagonal are zero ---
upper_elements = L[np.triu_indices(4, k=1)]
print("\nElements above diagonal in L:", upper_elements.round(10))
print("L is lower triangular?", np.allclose(upper_elements, 0))

# --- Verify L @ L.T reconstructs A ---
A_reconstructed = L @ L.T
print("\n--- Verification: L @ L.T = A ---")
print("L @ L.T:\n", A_reconstructed.round(6))
print("\nOriginal A:\n", A)
print("\nMatches A? (np.allclose):", np.allclose(A, A_reconstructed))

# --- Verify diagonal of L is positive ---
print("\nDiagonal of L (must all be > 0):", np.diag(L).round(6))
print("All positive?", np.all(np.diag(L) > 0))

# --- Use case: solving A @ x = b efficiently via Cholesky ---
# A @ x = b  →  L @ L.T @ x = b
# Step 1: solve L @ y = b   (forward substitution)
# Step 2: solve L.T @ x = y (back substitution)
b = np.array([10.0, 20.0, 15.0, 25.0])

y = np.linalg.solve(L,   b)             # forward substitution
x_chol = np.linalg.solve(L.T, y)        # back substitution

x_direct = np.linalg.solve(A, b)        # direct solve for comparison

print("\n--- Solving A @ x = b via Cholesky ---")
print("x via Cholesky  :", x_chol.round(6))
print("x via direct    :", x_direct.round(6))
print("Both match?", np.allclose(x_chol, x_direct))

# --- What happens with a non-SPD matrix ---
print("\n--- Cholesky on non-SPD matrix ---")
C = np.array([[1, 2], [3, 4]], dtype=float)   # not symmetric, not SPD
try:
    np.linalg.cholesky(C)
except np.linalg.LinAlgError as e:
    print(f"LinAlgError: {e}")

# --- Operation count comparison (printed summary) ---
print("\n--- Cholesky vs LU efficiency ---")
print("LU decomposition : ~(2/3) n³ flops  (general matrices)")
print("Cholesky         : ~(1/3) n³ flops  (SPD matrices only)")
print("Cholesky is ~2x faster than LU and uses half the storage")
print("(only L stored, not both L and U)")

B:
 [[5. 2. 4. 4.]
 [5. 2. 1. 2.]
 [3. 3. 1. 5.]
 [1. 5. 1. 4.]]

A = B.T @ B  (SPD matrix):
 [[60. 34. 29. 49.]
 [34. 42. 18. 47.]
 [29. 18. 19. 27.]
 [49. 47. 27. 61.]]

Is symmetric?        True
Eigenvalues:         [  1.9678   4.2896  19.6768 156.0659]
Is positive definite (all λ > 0)? True

L (lower triangular), shape (4, 4):
 [[7.745967 0.       0.       0.      ]
 [4.389381 4.767949 0.       0.      ]
 [3.743884 0.328583 2.208023 0.      ]
 [6.325873 4.03388  0.901803 1.974309]]

Elements above diagonal in L: [0. 0. 0. 0. 0. 0.]
L is lower triangular? True

--- Verification: L @ L.T = A ---
L @ L.T:
 [[60. 34. 29. 49.]
 [34. 42. 18. 47.]
 [29. 18. 19. 27.]
 [49. 47. 27. 61.]]

Original A:
 [[60. 34. 29. 49.]
 [34. 42. 18. 47.]
 [29. 18. 19. 27.]
 [49. 47. 27. 61.]]

Matches A? (np.allclose): True

Diagonal of L (must all be > 0): [7.745967 4.767949 2.208023 1.974309]
All positive? True

--- Solving A @ x = b via Cholesky ---
x via Cholesky  : [-1.065352  0.299757  1.782339  0.24

### How it works

- **Cholesky decomposition** splits a symmetric positive definite (SPD) matrix A into `L @ L.T`, where L is lower triangular with positive diagonal entries — think of it as a "square root" of a matrix.
- A matrix is **SPD** when it is symmetric (`A = A.T`) and all eigenvalues are positive. `B.T @ B` always produces one because it encodes squared distances, which are never negative.
- Solving `A @ x = b` via Cholesky uses two cheap triangular solves (forward then back substitution) instead of full Gaussian elimination.
- **Why Cholesky over LU?**
  - LU works on any square matrix and costs ~`(2/3)n³` flops
  - Cholesky exploits symmetry, costs only ~`(1/3)n³` flops — roughly **2× faster**
  - Cholesky stores only L (one triangle), LU stores both L and U — **half the memory**
  - Cholesky is also more numerically stable for SPD matrices because no pivoting is needed
- Cholesky appears constantly in ML: multivariate Gaussian sampling, Kalman filters, and Gaussian process inference all rely on it.

**Key takeaway:** Use Cholesky instead of LU whenever your matrix is symmetric positive definite — it is 2× faster, uses half the memory, and is numerically more stable, but it will raise `LinAlgError` if the matrix is not SPD.

### Q19. Matrix Power
Use `np.linalg.matrix_power(A, n)` to compute a matrix raised to an integer power. Demonstrate for `n=0` (identity), `n=1`, `n=2`, and `n=-1` (inverse). Verify that `matrix_power(A, -1)` equals `np.linalg.inv(A)`.
 

In [18]:
# Q19: Matrix power — n=0, 1, 2, -1 and verification

import numpy as np

A = np.array([[2, 1, 0],
              [1, 3, 1],
              [0, 1, 2]], dtype=float)

print("Matrix A:\n", A)
print(f"det(A) = {np.linalg.det(A):.4f}  (non-zero → invertible)\n")

# --- n = 0: any invertible matrix to the power 0 = Identity ---
A0 = np.linalg.matrix_power(A, 0)
print("A⁰  (n=0):\n", A0)
print("Equals Identity?", np.allclose(A0, np.eye(3)), "\n")

# --- n = 1: matrix unchanged ---
A1 = np.linalg.matrix_power(A, 1)
print("A¹  (n=1):\n", A1)
print("Equals A?", np.allclose(A1, A), "\n")

# --- n = 2: same as A @ A ---
A2        = np.linalg.matrix_power(A, 2)
A2_manual = A @ A
print("A²  (n=2):\n", A2)
print("Equals A @ A?", np.allclose(A2, A2_manual), "\n")

# --- n = 3: same as A @ A @ A ---
A3        = np.linalg.matrix_power(A, 3)
A3_manual = A @ A @ A
print("A³  (n=3):\n", A3)
print("Equals A @ A @ A?", np.allclose(A3, A3_manual), "\n")

# --- n = -1: matrix inverse ---
A_neg1  = np.linalg.matrix_power(A, -1)
A_inv   = np.linalg.inv(A)
print("A⁻¹ (n=-1) via matrix_power:\n", A_neg1.round(6))
print("\nA⁻¹ via np.linalg.inv:\n",      A_inv.round(6))
print("\nBoth equal?", np.allclose(A_neg1, A_inv))

# --- Verify inverse property: A @ A⁻¹ = I ---
print("\n--- Inverse verification ---")
print("A @ A⁻¹:\n", (A @ A_neg1).round(10))
print("Equals Identity?", np.allclose(A @ A_neg1, np.eye(3)))

# --- n = -2: same as inv(A) @ inv(A) ---
A_neg2        = np.linalg.matrix_power(A, -2)
A_neg2_manual = np.linalg.inv(A) @ np.linalg.inv(A)
print("\nA⁻² (n=-2) via matrix_power:\n",  A_neg2.round(6))
print("Equals inv(A) @ inv(A)?", np.allclose(A_neg2, A_neg2_manual))

# --- Eigenvalue connection: eigenvalues of Aⁿ = λⁿ ---
print("\n--- Eigenvalue connection ---")
eigenvalues_A  = np.linalg.eigh(A)[0]
eigenvalues_A3 = np.linalg.eigh(A3)[0]
print("Eigenvalues of A  :", eigenvalues_A.round(4))
print("Eigenvalues of A³ :", eigenvalues_A3.round(4))
print("λ³ for each λ     :", (eigenvalues_A ** 3).round(4))
print("Match?", np.allclose(np.sort(eigenvalues_A3),
                             np.sort(eigenvalues_A ** 3)))

# --- Summary table ---
print("\n--- Summary ---")
print(f"{'Power':<8} {'Meaning':<35} {'Verified'}")
print("-" * 55)
cases = [
    ("n=0",  "Identity matrix (A⁰ = I)",          np.allclose(A0, np.eye(3))),
    ("n=1",  "Matrix itself  (A¹ = A)",            np.allclose(A1, A)),
    ("n=2",  "A @ A",                              np.allclose(A2, A2_manual)),
    ("n=3",  "A @ A @ A",                          np.allclose(A3, A3_manual)),
    ("n=-1", "Inverse        (A⁻¹ = inv(A))",      np.allclose(A_neg1, A_inv)),
    ("n=-2", "inv(A) @ inv(A)",                    np.allclose(A_neg2, A_neg2_manual)),
]
for power, meaning, verified in cases:
    print(f"{power:<8} {meaning:<35} {verified}")

Matrix A:
 [[2. 1. 0.]
 [1. 3. 1.]
 [0. 1. 2.]]
det(A) = 8.0000  (non-zero → invertible)

A⁰  (n=0):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Equals Identity? True 

A¹  (n=1):
 [[2. 1. 0.]
 [1. 3. 1.]
 [0. 1. 2.]]
Equals A? True 

A²  (n=2):
 [[ 5.  5.  1.]
 [ 5. 11.  5.]
 [ 1.  5.  5.]]
Equals A @ A? True 

A³  (n=3):
 [[15. 21.  7.]
 [21. 43. 21.]
 [ 7. 21. 15.]]
Equals A @ A @ A? True 

A⁻¹ (n=-1) via matrix_power:
 [[ 0.625 -0.25   0.125]
 [-0.25   0.5   -0.25 ]
 [ 0.125 -0.25   0.625]]

A⁻¹ via np.linalg.inv:
 [[ 0.625 -0.25   0.125]
 [-0.25   0.5   -0.25 ]
 [ 0.125 -0.25   0.625]]

Both equal? True

--- Inverse verification ---
A @ A⁻¹:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Equals Identity? True

A⁻² (n=-2) via matrix_power:
 [[ 0.46875 -0.3125   0.21875]
 [-0.3125   0.375   -0.3125 ]
 [ 0.21875 -0.3125   0.46875]]
Equals inv(A) @ inv(A)? True

--- Eigenvalue connection ---
Eigenvalues of A  : [1. 2. 4.]
Eigenvalues of A³ : [ 1.  8. 64.]
λ³ for each λ     : [ 1.  8. 64.]
Match? Tru

### How it works

- `np.linalg.matrix_power(A, n)` raises a square matrix to an integer power — like scalar exponentiation but for matrices.
- Key power rules (mirror scalar rules exactly):
  - `n=0` → Identity (anything to the power 0 = 1; for matrices, 1 = I)
  - `n=1` → A itself unchanged
  - `n=2` → `A @ A` (not element-wise squaring — full matrix multiplication)
  - `n=-1` → matrix inverse (undo A once)
  - `n=-2` → `inv(A) @ inv(A)` (undo A twice)
- **Eigenvalue shortcut:** if A has eigenvalue λ, then Aⁿ has eigenvalue λⁿ — NumPy uses this internally for efficiency via repeated squaring.
- `matrix_power(A, -1)` will raise `LinAlgError` for singular matrices, just like `np.linalg.inv`.
- Matrix powers appear in Markov chains (`Aⁿ` gives n-step transition probabilities) and graph theory (`A²[i,j]` counts paths of length 2 between nodes i and j).

**Key takeaway:** `matrix_power(A, n)` follows the same rules as scalar exponentiation — `n=0` gives Identity, `n=-1` gives the inverse — but always uses matrix multiplication (`@`), never element-wise.

### Q20. Computing the Condition Number
Use `np.linalg.cond(A)` to compute the condition number of several matrices (identity matrix, a well-conditioned matrix, and an ill-conditioned/near-singular matrix). Explain: a high condition number means small changes in `b` can cause large changes in the solution `x` of `Ax = b`.
 

In [19]:
# Q20: Condition number — well-conditioned vs ill-conditioned matrices

import numpy as np

np.random.seed(42)

# --- 1. Identity matrix: perfectly conditioned ---
I = np.eye(4)
cond_I = np.linalg.cond(I)
print("Identity matrix I:\n", I)
print(f"Condition number: {cond_I:.4f}  (perfect — cond=1 is the best possible)\n")

# --- 2. Well-conditioned matrix ---
W = np.array([[4, 1, 0],
              [1, 3, 1],
              [0, 1, 2]], dtype=float)
cond_W = np.linalg.cond(W)
print("Well-conditioned matrix W:\n", W)
print(f"Condition number: {cond_W:.4f}  (small → reliable solutions)\n")

# --- 3. Ill-conditioned matrix: rows nearly linearly dependent ---
# Small perturbation between rows makes det ≈ 0
ill = np.array([[1.000, 2.000],
                [1.001, 2.002]], dtype=float)   # row2 ≈ row1 × 1.001
cond_ill = np.linalg.cond(ill)
print("Ill-conditioned 2x2 matrix:\n", ill)
print(f"Condition number: {cond_ill:.2f}  (huge → tiny input change → large output change)\n")

# --- 4. Near-singular 3x3 matrix ---
S = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9.001]], dtype=float)   # nearly singular (det ≈ 0)
cond_S = np.linalg.cond(S)
print("Near-singular 3x3 matrix:\n", S)
print(f"Condition number: {cond_S:.2e}  (extremely large → numerically unstable)\n")

# --- Condition number = ratio of largest to smallest singular value ---
print("--- Condition number = σ_max / σ_min (via SVD) ---")
for name, M in [("Identity", I), ("Well-cond W", W),
                ("Ill-cond", ill), ("Near-singular", S)]:
    sv      = np.linalg.svd(M, compute_uv=False)   # singular values only
    cond_sv = sv[0] / sv[-1]                        # σ_max / σ_min
    cond_np = np.linalg.cond(M)
    print(f"  {name:<16} σ_max={sv[0]:.4f}  σ_min={sv[-1]:.2e}  "
          f"ratio={cond_sv:.4f}  np.cond={cond_np:.4f}  match={np.isclose(cond_sv, cond_np)}")

# --- Concrete demonstration: small change in b → large change in x ---
print("\n--- Sensitivity demo: ill-conditioned system ---")
A = ill
b1 = np.array([3.000, 3.003])          # original b
b2 = np.array([3.000, 3.004])          # tiny change: only 0.001 difference

x1 = np.linalg.solve(A, b1)
x2 = np.linalg.solve(A, b2)

b_change = np.linalg.norm(b2 - b1)
x_change = np.linalg.norm(x2 - x1)

print(f"b1 = {b1}  →  x1 = {x1.round(4)}")
print(f"b2 = {b2}  →  x2 = {x2.round(4)}")
print(f"\n||Δb|| = {b_change:.4f}  (tiny change in b)")
print(f"||Δx|| = {x_change:.4f}  (huge change in x!)")
print(f"Amplification factor: {x_change / b_change:.1f}x  ≈  cond(A) = {cond_ill:.1f}")

# --- Well-conditioned comparison ---
print("\n--- Sensitivity demo: well-conditioned system ---")
b3 = np.array([3.0, 2.0, 1.0])
b4 = np.array([3.001, 2.0, 1.0])      # same tiny change

x3 = np.linalg.solve(W, b3)
x4 = np.linalg.solve(W, b4)

b_chg = np.linalg.norm(b4 - b3)
x_chg = np.linalg.norm(x4 - x3)

print(f"||Δb|| = {b_chg:.4f}  →  ||Δx|| = {x_chg:.6f}  (small, controlled change)")
print(f"Amplification factor: {x_chg / b_chg:.4f}x  ≈  cond(W) = {cond_W:.4f}")

# --- Rule of thumb: digits of accuracy lost ---
print("\n--- Rule of thumb: log10(cond) ≈ decimal digits of accuracy lost ---")
for name, cond in [("Identity",     cond_I),
                   ("Well-cond W",  cond_W),
                   ("Ill-cond",     cond_ill),
                   ("Near-singular",cond_S)]:
    digits_lost = np.log10(cond)
    print(f"  {name:<16} cond={cond:.2e}  digits lost ≈ {digits_lost:.1f}")

Identity matrix I:
 [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
Condition number: 1.0000  (perfect — cond=1 is the best possible)

Well-conditioned matrix W:
 [[4. 1. 0.]
 [1. 3. 1.]
 [0. 1. 2.]]
Condition number: 3.7321  (small → reliable solutions)

Ill-conditioned 2x2 matrix:
 [[1.    2.   ]
 [1.001 2.002]]
Condition number: inf  (huge → tiny input change → large output change)

Near-singular 3x3 matrix:
 [[1.    2.    3.   ]
 [4.    5.    6.   ]
 [7.    8.    9.001]]
Condition number: 1.01e+05  (extremely large → numerically unstable)

--- Condition number = σ_max / σ_min (via SVD) ---
  Identity         σ_max=1.0000  σ_min=1.00e+00  ratio=1.0000  np.cond=1.0000  match=True
  Well-cond W      σ_max=4.7321  σ_min=1.27e+00  ratio=3.7321  np.cond=3.7321  match=True
  Ill-cond         σ_max=3.1639  σ_min=0.00e+00  ratio=inf  np.cond=inf  match=True
  Near-singular    σ_max=16.8487  σ_min=1.67e-04  ratio=101072.2649  np.cond=101072.2649  match=True

--- Sensitivity demo

/tmp/ipykernel_10050/990121463.py:42: RuntimeWarning: divide by zero encountered in scalar divide
  cond_sv = sv[0] / sv[-1]                        # σ_max / σ_min


LinAlgError: Singular matrix

In [20]:
# Q20: Condition number — well-conditioned vs ill-conditioned matrices (FIXED)

import numpy as np

np.random.seed(42)

# --- 1. Identity matrix: perfectly conditioned ---
I = np.eye(4)
cond_I = np.linalg.cond(I)
print("Identity matrix I:\n", I)
print(f"Condition number: {cond_I:.4f}  (perfect — cond=1 is the best possible)\n")

# --- 2. Well-conditioned matrix ---
W = np.array([[4, 1, 0],
              [1, 3, 1],
              [0, 1, 2]], dtype=float)
cond_W = np.linalg.cond(W)
print("Well-conditioned matrix W:\n", W)
print(f"Condition number: {cond_W:.4f}  (small → reliable solutions)\n")

# --- 3. Ill-conditioned matrix: rows NEARLY (not exactly) linearly dependent ---
# Key fix: 2.003 instead of 2.002 — det ≠ 0 but still tiny → huge cond number
ill = np.array([[1.000, 2.000],
                [1.001, 2.003]], dtype=float)
cond_ill = np.linalg.cond(ill)
print("Ill-conditioned 2x2 matrix:\n", ill)
print(f"det(ill) = {np.linalg.det(ill):.6f}  (non-zero but tiny)")
print(f"Condition number: {cond_ill:.2f}  (huge → tiny input change → large output change)\n")

# --- 4. Near-singular 3x3 matrix ---
S = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9.001]], dtype=float)
cond_S = np.linalg.cond(S)
print("Near-singular 3x3 matrix:\n", S)
print(f"Condition number: {cond_S:.2e}  (extremely large → numerically unstable)\n")

# --- Condition number = σ_max / σ_min (via SVD) ---
print("--- Condition number = σ_max / σ_min ---")
for name, M in [("Identity", I), ("Well-cond W", W),
                ("Ill-cond", ill), ("Near-singular", S)]:
    sv      = np.linalg.svd(M, compute_uv=False)
    cond_sv = sv[0] / sv[-1]
    cond_np = np.linalg.cond(M)
    print(f"  {name:<16} σ_max={sv[0]:.4f}  σ_min={sv[-1]:.2e}  "
          f"ratio={cond_sv:.2f}  np.cond={cond_np:.2f}  match={np.isclose(cond_sv, cond_np)}")

# --- Concrete demo: small change in b → large change in x (ill-conditioned) ---
print("\n--- Sensitivity demo: ill-conditioned system ---")
b1 = np.array([3.000, 3.003])      # original b
b2 = np.array([3.000, 3.004])      # tiny nudge: Δb = 0.001 in one element

x1 = np.linalg.solve(ill, b1)
x2 = np.linalg.solve(ill, b2)

b_change = np.linalg.norm(b2 - b1)
x_change = np.linalg.norm(x2 - x1)

print(f"b1 = {b1}  →  x1 = {x1.round(4)}")
print(f"b2 = {b2}  →  x2 = {x2.round(4)}")
print(f"\n||Δb|| = {b_change:.4f}  (tiny change in b)")
print(f"||Δx|| = {x_change:.4f}  (huge change in x!)")
print(f"Amplification: {x_change / b_change:.1f}x  ≈  cond(A) = {cond_ill:.1f}")

# --- Same sized Δb on well-conditioned system — barely moves x ---
print("\n--- Sensitivity demo: well-conditioned system ---")
b3 = np.array([3.000, 2.000, 1.000])
b4 = np.array([3.001, 2.000, 1.000])   # same tiny nudge

x3 = np.linalg.solve(W, b3)
x4 = np.linalg.solve(W, b4)

b_chg = np.linalg.norm(b4 - b3)
x_chg = np.linalg.norm(x4 - x3)

print(f"||Δb|| = {b_chg:.4f}  →  ||Δx|| = {x_chg:.6f}  (small, controlled change)")
print(f"Amplification: {x_chg / b_chg:.4f}x  ≈  cond(W) = {cond_W:.4f}")

# --- Rule of thumb: digits of accuracy lost = log10(cond) ---
print("\n--- Rule of thumb: log10(cond) ≈ decimal digits of accuracy lost ---")
for name, cond in [("Identity",      cond_I),
                   ("Well-cond W",   cond_W),
                   ("Ill-cond",      cond_ill),
                   ("Near-singular", cond_S)]:
    digits_lost = np.log10(cond)
    print(f"  {name:<16} cond={cond:.2e}  digits lost ≈ {digits_lost:.1f}")

Identity matrix I:
 [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
Condition number: 1.0000  (perfect — cond=1 is the best possible)

Well-conditioned matrix W:
 [[4. 1. 0.]
 [1. 3. 1.]
 [0. 1. 2.]]
Condition number: 3.7321  (small → reliable solutions)

Ill-conditioned 2x2 matrix:
 [[1.    2.   ]
 [1.001 2.003]]
det(ill) = 0.001000  (non-zero but tiny)
Condition number: 10014.01  (huge → tiny input change → large output change)

Near-singular 3x3 matrix:
 [[1.    2.    3.   ]
 [4.    5.    6.   ]
 [7.    8.    9.001]]
Condition number: 1.01e+05  (extremely large → numerically unstable)

--- Condition number = σ_max / σ_min ---
  Identity         σ_max=1.0000  σ_min=1.00e+00  ratio=1.00  np.cond=1.00  match=True
  Well-cond W      σ_max=4.7321  σ_min=1.27e+00  ratio=3.73  np.cond=3.73  match=True
  Ill-cond         σ_max=3.1645  σ_min=3.16e-04  ratio=10014.01  np.cond=10014.01  match=True
  Near-singular    σ_max=16.8487  σ_min=1.67e-04  ratio=101072.26  np.cond=101072.26

### Bug fix

The original `ill` matrix had `[1.001, 2.002]` as its second row — exactly `1.001 ×` the first row, making `det = 0` (truly singular). `np.linalg.solve` correctly refused to proceed.

**Fix:** change `2.002` to `2.003` so the rows are *nearly* but not exactly dependent. Now `det ≈ 0.001` (tiny but non-zero) and `cond ≈ 10,000` — the matrix is solvable but highly sensitive to perturbations, which is exactly what ill-conditioned means.

**Key lesson:** ill-conditioned ≠ singular. Singular means `det = 0` exactly (no solution). Ill-conditioned means `det ≈ 0` (a solution exists but tiny input noise produces huge output noise).

### How it works

- The **condition number** measures how sensitive a linear system `Ax = b` is to small changes in b or A. It is always ≥ 1, with 1 being perfectly stable.
- Mathematically it equals `σ_max / σ_min` — the ratio of the largest to smallest singular value. A tiny σ_min means the matrix nearly collapses a direction to zero, making it impossible to reliably reverse.
- Think of it like a magnifying glass for errors: a condition number of 1000 means a 0.1% change in b can cause up to a 100% change in x.
- **Rule of thumb:** `log10(cond(A))` ≈ number of decimal digits of accuracy lost in the solution. A condition number of 10⁸ on a 64-bit float (≈15 significant digits) leaves only 7 reliable digits.
- Key ranges:
  - `cond ≈ 1` → perfectly conditioned (identity)
  - `cond < 100` → well-conditioned, trustworthy solution
  - `cond > 10⁶` → ill-conditioned, results may be unreliable
  - `cond → ∞` → singular matrix, no solution

**Key takeaway:** Always check `np.linalg.cond(A)` before trusting a linear system solution — a large condition number means small noise in your data can produce wildly wrong answers, regardless of which solver you use.

### Q21. Principal Component Analysis (PCA) — From Scratch
Given a `(50, 3)` dataset of random data:
1. Center the data (subtract mean of each column)
2. Compute the covariance matrix using `np.cov`
3. Get eigenvalues and eigenvectors of the covariance matrix
4. Sort by descending eigenvalue
5. Project data onto the top 2 principal components
This is the mathematical core of PCA using only `np.linalg`. Print the explained variance ratio for each component.

In [21]:
# Q21: PCA from scratch using np.linalg — center, covariance, eigen, project

import numpy as np

np.random.seed(0)

# --- Generate (50, 3) dataset with intentional structure ---
# Make feature 3 a noisy combo of features 1 and 2 → clear dominant components
X_raw = np.random.randn(50, 3)
X_raw[:, 2] = 0.8 * X_raw[:, 0] + 0.6 * X_raw[:, 1] + 0.2 * np.random.randn(50)

print("Original data X_raw (shape {})".format(X_raw.shape))
print("First 5 rows:\n", X_raw[:5].round(4))

# --- Step 1: Center the data (subtract column means) ---
col_means = np.mean(X_raw, axis=0)         # shape (3,) — one mean per feature
X_centered = X_raw - col_means             # broadcasts across all 50 rows

print("\n--- Step 1: Centering ---")
print("Column means (before):", col_means.round(6))
print("Column means (after) :", np.mean(X_centered, axis=0).round(10))  # ≈ zero

# --- Step 2: Covariance matrix ---
# np.cov expects features as ROWS → transpose input (3, 50)
# result is (3, 3): covariance between every pair of features
cov_matrix = np.cov(X_centered.T)         # shape (3, 3)

print("\n--- Step 2: Covariance matrix (3×3) ---")
print(cov_matrix.round(4))
print("Symmetric?", np.allclose(cov_matrix, cov_matrix.T))

# --- Step 3: Eigenvalues and eigenvectors of covariance matrix ---
# eigh preferred: covariance matrix is symmetric → real, sorted eigenvalues
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

print("\n--- Step 3: Eigendecomposition ---")
print("Eigenvalues  (ascending) :", eigenvalues.round(6))
print("Eigenvectors (columns):\n", eigenvectors.round(6))

# --- Step 4: Sort by DESCENDING eigenvalue ---
sorted_idx  = np.argsort(eigenvalues)[::-1]      # reverse ascending → descending
eigenvalues  = eigenvalues[sorted_idx]
eigenvectors = eigenvectors[:, sorted_idx]        # reorder columns to match

print("\n--- Step 4: Sorted descending ---")
print("Eigenvalues  (descending):", eigenvalues.round(6))
print("Eigenvectors (columns):\n", eigenvectors.round(6))

# --- Explained variance ratio ---
total_variance      = np.sum(eigenvalues)
explained_ratio     = eigenvalues / total_variance
explained_cumulative = np.cumsum(explained_ratio)

print("\n--- Explained Variance ---")
print(f"Total variance: {total_variance:.4f}")
for i, (ev, ratio, cumul) in enumerate(zip(eigenvalues, explained_ratio, explained_cumulative)):
    print(f"  PC{i+1}: eigenvalue={ev:.4f}  "
          f"explained={ratio*100:.2f}%  cumulative={cumul*100:.2f}%")

# --- Step 5: Project data onto top 2 principal components ---
# Principal components = top-2 eigenvectors as columns → shape (3, 2)
W = eigenvectors[:, :2]                   # projection matrix, shape (3, 2)

# Project: (50, 3) @ (3, 2) = (50, 2)
X_projected = X_centered @ W

print("\n--- Step 5: Projection ---")
print(f"Projection matrix W shape : {W.shape}  (3 features → 2 PCs)")
print(f"Projected data shape      : {X_projected.shape}  (50 samples, 2 PCs)")
print("First 5 rows of projected data:\n", X_projected[:5].round(4))

# --- Verify: projected components are uncorrelated (cov ≈ diagonal) ---
cov_projected = np.cov(X_projected.T)
print("\n--- Verification: covariance of projected data (should be diagonal) ---")
print(cov_projected.round(6))
print("Off-diagonal ≈ 0?", np.allclose(cov_projected[0, 1], 0, atol=1e-10))

# --- Verify: variance along each PC = its eigenvalue ---
print("\n--- Variance along each PC = eigenvalue ---")
for i in range(2):
    var_pc = np.var(X_projected[:, i], ddof=1)
    print(f"  PC{i+1}: var={var_pc:.6f}  eigenvalue={eigenvalues[i]:.6f}  "
          f"match={np.isclose(var_pc, eigenvalues[i])}")

Original data X_raw (shape (50, 3))
First 5 rows:
 [[ 1.7641  0.4002  1.6377]
 [ 2.2409  1.8676  3.2559]
 [ 0.9501 -0.1514  0.5203]
 [ 0.4106  0.144   0.2496]
 [ 0.761   0.1217  0.6621]]

--- Step 1: Centering ---
Column means (before): [0.279355 0.212079 0.332229]
Column means (after) : [ 0. -0.  0.]

--- Step 2: Covariance matrix (3×3) ---
[[ 1.1419 -0.013   0.8799]
 [-0.013   1.0237  0.5819]
 [ 0.8799  0.5819  1.0606]]
Symmetric? True

--- Step 3: Eigendecomposition ---
Eigenvalues  (ascending) : [0.020625 1.07162  2.134023]
Eigenvectors (columns):
 [[ 0.563233 -0.550259 -0.616427]
 [ 0.420063  0.833097 -0.359857]
 [-0.711558  0.056255 -0.700372]]

--- Step 4: Sorted descending ---
Eigenvalues  (descending): [2.134023 1.07162  0.020625]
Eigenvectors (columns):
 [[-0.616427 -0.550259  0.563233]
 [-0.359857  0.833097  0.420063]
 [-0.700372  0.056255 -0.711558]]

--- Explained Variance ---
Total variance: 3.2263
  PC1: eigenvalue=2.1340  explained=66.15%  cumulative=66.15%
  PC2: eigen

### How it works

- **PCA** finds the directions of maximum variance in data and projects onto them, reducing dimensions while keeping the most information.
- The five steps mirror what `sklearn.decomposition.PCA` does internally:
  - **Center:** remove the mean so variance is measured around zero, not the data's origin
  - **Covariance matrix:** captures how much each pair of features varies together — symmetric, so `eigh` is used
  - **Eigendecomposition:** eigenvectors of the covariance matrix are the principal component directions; eigenvalues measure variance along each direction
  - **Sort descending:** PC1 always explains the most variance, PC2 the second most
  - **Project:** multiply centered data by the top-k eigenvectors to get the low-dimensional representation
- **Explained variance ratio** = each eigenvalue divided by total variance — tells you what fraction of information each PC retains.
- After projection, the two PCs are guaranteed **uncorrelated** (off-diagonal covariance ≈ 0) and the variance along each PC exactly equals its eigenvalue.

**Key takeaway:** PCA is eigendecomposition of the covariance matrix — eigenvalues measure how much variance each direction holds, and projecting onto the top-k eigenvectors gives the best possible k-dimensional summary of the data.

### Q22. Solving Multiple Right-Hand Sides
`np.linalg.solve` handles only one right-hand side vector `b`. Solve the same matrix equation `Ax = B` for multiple right-hand side columns simultaneously by passing a 2D matrix `B` (each column is a separate `b`). Verify each column of the solution independently.

In [27]:
# Q22: Solving Ax = B for multiple right-hand sides simultaneously

import numpy as np

# --- Coefficient matrix A (same for all systems) ---
A = np.array([[2, 1, 3],
              [1, 4, 2],
              [3, 2, 5]], dtype=float)

# --- B: each column is a separate right-hand side ---
# Solving 4 systems at once: Ax = b1, Ax = b2, Ax = b3, Ax = b4
B = np.array([[1,  5,  9,  2],
              [2,  6, 10,  0],
              [3,  7, 11,  4]], dtype=float)   # shape (3, 4)

print("Coefficient matrix A (shape {}):\n".format(A.shape), A)
print("\nRight-hand side matrix B (shape {}):".format(B.shape))
print("(each column = one independent system)\n", B)

# --- Solve all 4 systems simultaneously ---
# X shape: (3, 4) — each column of X is solution for corresponding column of B
X = np.linalg.solve(A, B)

print("Solution matrix X (shape {}):\n".format(X.shape), X.round(6))
print("Each column of X solves Ax = b_i for that column of B\n")

# --- Verify each column independently ---
print("--- Column-by-column verification: A @ x_i ≈ b_i ---")
all_match = True
for i in range(B.shape[1]):
    x_i   = X[:, i]                    # solution for system i
    b_i   = B[:, i]                    # right-hand side for system i
    Ax_i  = A @ x_i                    # should reconstruct b_i
    match = np.allclose(Ax_i, b_i)
    all_match = all_match and match
    print(f"  System {i+1}: A @ x{i+1} = {Ax_i.round(4)}  |  b{i+1} = {b_i}  |  match: {match}")

print(f"\nAll {B.shape[1]} systems solved correctly? {all_match}")

# --- Verify via full matrix: A @ X ≈ B ---
print("\n--- Full matrix verification: A @ X ≈ B ---")
print("A @ X:\n", (A @ X).round(6))
print("\nOriginal B:\n", B)
print("\nnp.allclose(A @ X, B):", np.allclose(A @ X, B))

# --- Compare: solving column-by-column vs all at once ---
print("\n--- Efficiency: one call vs four separate calls ---")
X_colbycol = np.column_stack([np.linalg.solve(A, B[:, i])
                               for i in range(B.shape[1])])
print("Column-by-column result:\n", X_colbycol.round(6))
print("Matches single call?", np.allclose(X, X_colbycol))

# --- Practical use case: multiple load cases in structural analysis ---
print("\n--- Practical context ---")
print("Same stiffness matrix A, four different load vectors b1..b4")
print("One np.linalg.solve call returns all four displacement solutions")
print(f"Shape in : A{A.shape}, B{B.shape}")
print(f"Shape out: X{X.shape}  — one solution column per load case")

Coefficient matrix A (shape (3, 3)):
 [[2. 1. 3.]
 [1. 4. 2.]
 [3. 2. 5.]]

Right-hand side matrix B (shape (3, 4)):
(each column = one independent system)
 [[ 1.  5.  9.  2.]
 [ 2.  6. 10.  0.]
 [ 3.  7. 11.  4.]]
Solution matrix X (shape (3, 4)):
 [[-4.        5.333333 14.666667 -2.666667]
 [-0.        1.333333  2.666667 -0.666667]
 [ 3.       -2.333333 -7.666667  2.666667]]
Each column of X solves Ax = b_i for that column of B

--- Column-by-column verification: A @ x_i ≈ b_i ---
  System 1: A @ x1 = [1. 2. 3.]  |  b1 = [1. 2. 3.]  |  match: True
  System 2: A @ x2 = [5. 6. 7.]  |  b2 = [5. 6. 7.]  |  match: True
  System 3: A @ x3 = [ 9. 10. 11.]  |  b3 = [ 9. 10. 11.]  |  match: True
  System 4: A @ x4 = [ 2. -0.  4.]  |  b4 = [2. 0. 4.]  |  match: True

All 4 systems solved correctly? True

--- Full matrix verification: A @ X ≈ B ---
A @ X:
 [[ 1.  5.  9.  2.]
 [ 2.  6. 10. -0.]
 [ 3.  7. 11.  4.]]

Original B:
 [[ 1.  5.  9.  2.]
 [ 2.  6. 10.  0.]
 [ 3.  7. 11.  4.]]

np.allclo

### How it works

- `np.linalg.solve(A, B)` accepts B as either a 1D vector (one system) or a **2D matrix** (many systems). Each column of B is treated as an independent right-hand side.
- The result X has the same shape as B — column i of X solves `A @ X[:,i] = B[:,i]`.
- This is far more efficient than looping: NumPy performs a **single LU decomposition** of A and reuses it for all right-hand sides, instead of factorising A once per column.
- Think of it like a factory: factorising A is the expensive setup; applying it to each b column is cheap. Doing it in one call amortises the setup cost across all systems.
- The full-matrix verification `np.allclose(A @ X, B)` confirms all systems at once — cleaner than checking each column individually.

**Key takeaway:** Pass a 2D matrix B to `np.linalg.solve` to solve multiple systems simultaneously — one LU factorisation shared across all right-hand sides is faster and cleaner than looping over columns.

### Q23. Computing Pairwise Distances via Linear Algebra
Given a matrix `X` of shape `(n, d)`, compute the pairwise squared Euclidean distance matrix entirely using matrix operations:
`D[i,j] = ||x_i - x_j||^2 = ||x_i||^2 + ||x_j||^2 - 2 * x_i · x_j`
 
Implement this without any loops, using only `np.dot`, broadcasting, and `np.sum`. Verify correctness for a small case by also computing brute-force.

In [28]:
# Q23: Pairwise squared Euclidean distances — vectorised vs brute-force

import numpy as np

np.random.seed(1)
X = np.random.randint(0, 6, size=(5, 3)).astype(float)  # 5 points in 3D space

print("Data matrix X (5 points in 3D, shape {}):\n".format(X.shape), X)

# --- Vectorised: D[i,j] = ||xi||² + ||xj||² - 2 * xi·xj ---

# Step 1: squared L2 norm of each row → shape (5,)
row_sq_norms = np.sum(X ** 2, axis=1)               # ||xi||² for each i
print("\nRow squared norms ||xi||²:", row_sq_norms)

# Step 2: cross-term matrix 2 * X @ X.T → shape (5, 5)
# entry [i,j] = 2 * xi · xj
cross = 2 * (X @ X.T)                               # shape (5, 5)

# Step 3: broadcast norms into (5, 5) distance matrix
# row_sq_norms[:, None] → column vector (5,1) adds ||xi||² to each row i
# row_sq_norms[None, :] → row vector    (1,5) adds ||xj||² to each col j
D_vectorised = row_sq_norms[:, None] + row_sq_norms[None, :] - cross

print("\n--- Vectorised pairwise squared distance matrix D (shape {}) ---".format(D_vectorised.shape))
print(D_vectorised.round(4))

# --- Brute-force: explicit double loop for verification ---
n = X.shape[0]
D_brute = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        diff = X[i] - X[j]                          # shape (3,)
        D_brute[i, j] = np.dot(diff, diff)          # ||xi - xj||²

print("\n--- Brute-force pairwise squared distance matrix ---")
print(D_brute.round(4))

# --- Verify both methods match ---
print("\nVectorised == Brute-force? (np.allclose):", np.allclose(D_vectorised, D_brute))

# --- Sanity checks on the distance matrix ---
print("\n--- Sanity checks ---")
print("Diagonal all zero (distance to self)?  ", np.allclose(np.diag(D_vectorised), 0))
print("Symmetric D[i,j] == D[j,i]?           ", np.allclose(D_vectorised, D_vectorised.T))
print("All values non-negative?               ", np.all(D_vectorised >= -1e-10))

# --- Euclidean (non-squared) distances ---
D_euclidean = np.sqrt(np.maximum(D_vectorised, 0))  # clip tiny negatives from float noise
print("\n--- Euclidean distance matrix (non-squared) ---")
print(D_euclidean.round(4))

# --- Practical use: find nearest neighbour for each point ---
print("\n--- Nearest neighbour for each point (excluding self) ---")
D_no_self = D_euclidean.copy()
np.fill_diagonal(D_no_self, np.inf)                 # mask self-distance

nearest = np.argmin(D_no_self, axis=1)              # index of closest point
for i in range(n):
    j   = nearest[i]
    dist = D_euclidean[i, j]
    print(f"  Point {i} {X[i].astype(int)} → nearest: Point {j} {X[j].astype(int)}  dist={dist:.4f}")

# --- Performance intuition ---
print("\n--- Why vectorised beats loops ---")
print(f"Brute-force: {n}×{n} = {n*n} iterations, {n*n} np.dot calls")
print(f"Vectorised : 1 matrix multiply (X @ X.T) + 2 broadcasts — no Python loop")
print(f"For n=1000 points: brute-force = 1,000,000 iterations vs 1 matmul")

Data matrix X (5 points in 3D, shape (5, 3)):
 [[5. 3. 4.]
 [0. 1. 3.]
 [5. 0. 0.]
 [1. 4. 5.]
 [4. 1. 2.]]

Row squared norms ||xi||²: [50. 10. 25. 42. 21.]

--- Vectorised pairwise squared distance matrix D (shape (5, 5)) ---
[[ 0. 30. 25. 18.  9.]
 [30.  0. 35. 14. 17.]
 [25. 35.  0. 57.  6.]
 [18. 14. 57.  0. 27.]
 [ 9. 17.  6. 27.  0.]]

--- Brute-force pairwise squared distance matrix ---
[[ 0. 30. 25. 18.  9.]
 [30.  0. 35. 14. 17.]
 [25. 35.  0. 57.  6.]
 [18. 14. 57.  0. 27.]
 [ 9. 17.  6. 27.  0.]]

Vectorised == Brute-force? (np.allclose): True

--- Sanity checks ---
Diagonal all zero (distance to self)?   True
Symmetric D[i,j] == D[j,i]?            True
All values non-negative?                True

--- Euclidean distance matrix (non-squared) ---
[[0.     5.4772 5.     4.2426 3.    ]
 [5.4772 0.     5.9161 3.7417 4.1231]
 [5.     5.9161 0.     7.5498 2.4495]
 [4.2426 3.7417 7.5498 0.     5.1962]
 [3.     4.1231 2.4495 5.1962 0.    ]]

--- Nearest neighbour for each point (ex

### How it works

- The identity `||xi - xj||² = ||xi||² + ||xj||² - 2·xi·xj` rewrites subtraction as three cheaper operations: two norm computations and one dot product matrix.
- **Step 1** computes all row norms at once with `np.sum(X**2, axis=1)` — shape `(n,)`.
- **Step 2** computes all cross-terms in one matrix multiply `X @ X.T` — entry `[i,j]` = `xi·xj`, shape `(n,n)`.
- **Step 3** uses broadcasting to add norms: `row_sq_norms[:, None]` is shape `(n,1)` and broadcasts across columns; `row_sq_norms[None, :]` is shape `(1,n)` and broadcasts across rows — no loop needed.
- The resulting matrix must be **symmetric** (distance is the same both ways) and have **zeros on the diagonal** (distance to self = 0) — both are quick sanity checks.
- `np.maximum(D, 0)` before `sqrt` clips tiny negative values (e.g. `-1e-15`) caused by floating-point rounding before taking the square root.

**Key takeaway:** Replace `||xi - xj||²` with `||xi||² + ||xj||² - 2·xi·xj` to turn an O(n²) Python loop into a single matrix multiply plus two broadcasts — the standard trick used inside k-NN, k-means, and kernel methods for speed.

### Q24. Rotation Matrix Application
Write a function `rotate_2d(points, angle_degrees)` that takes an array of 2D points (shape `(n, 2)`) and rotates them all by a given angle using the 2D rotation matrix:
```
R = [[cos θ, -sin θ],
     [sin θ,  cos θ]]
```
Apply it to a unit circle (50 points) by 45 degrees. Verify the rotated points still lie on the unit circle (all norms ≈ 1).

In [29]:
# Q24: 2D rotation matrix — rotate points, verify unit circle preserved

import numpy as np

# --- Rotation function ---
def rotate_2d(points, angle_degrees):
    """
    Rotate an (n, 2) array of 2D points by angle_degrees anticlockwise.
    Returns rotated points of same shape (n, 2).
    """
    theta = np.radians(angle_degrees)          # convert degrees → radians

    # 2D rotation matrix R (2×2)
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])

    # (n,2) @ (2,2).T  =  (n,2)  — rotate every point in one matrix multiply
    return points @ R.T

# --- Build a unit circle: 50 evenly spaced points ---
angles   = np.linspace(0, 2 * np.pi, 50, endpoint=False)   # 50 angles 0..2π
circle   = np.column_stack([np.cos(angles),                 # x = cos(θ)
                             np.sin(angles)])                # y = sin(θ)

print("Unit circle points (shape {})".format(circle.shape))
print("First 5 points:\n", circle[:5].round(4))
print("All norms before rotation:",
      np.allclose(np.linalg.norm(circle, axis=1), 1.0))

# --- Rotate by 45 degrees ---
rotated_45 = rotate_2d(circle, 45)

print("\nRotated 45° points (shape {})".format(rotated_45.shape))
print("First 5 points:\n", rotated_45[:5].round(4))

# --- Verify: all rotated points still on unit circle (norm ≈ 1) ---
norms_after = np.linalg.norm(rotated_45, axis=1)
print("\nNorms after 45° rotation (first 10):", norms_after[:10].round(8))
print("All norms ≈ 1.0?", np.allclose(norms_after, 1.0))

# --- Verify rotation matrix properties ---
theta = np.radians(45)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

print("\n--- Rotation matrix R (45°) ---")
print(R.round(6))
print("det(R) =", round(np.linalg.det(R), 10),    "  (must be 1.0 — no scaling)")
print("R.T @ R = I?", np.allclose(R.T @ R, np.eye(2)), "  (R is orthogonal)")

# --- Demonstrate several rotation angles ---
print("\n--- Rotating point [1, 0] by common angles ---")
p = np.array([[1.0, 0.0]])             # point on positive x-axis
for deg in [0, 45, 90, 180, 270, 360]:
    rotated = rotate_2d(p, deg)
    print(f"  {deg:>3}° → {rotated[0].round(4)}")

# --- Verify: rotating 360° returns to original ---
rotated_360 = rotate_2d(circle, 360)
print("\nRotate 360° returns to original?", np.allclose(rotated_360, circle))

# --- Verify: two 45° rotations = one 90° rotation ---
rotated_45_twice = rotate_2d(rotate_2d(circle, 45), 45)
rotated_90_once  = rotate_2d(circle, 90)
print("45° twice == 90° once?", np.allclose(rotated_45_twice, rotated_90_once))

# --- Verify: rotation is reversible (rotate then un-rotate = original) ---
rotated   = rotate_2d(circle, 73)       # arbitrary angle
unrotated = rotate_2d(rotated, -73)     # rotate back by same angle
print("Rotate +73° then -73° = original?", np.allclose(unrotated, circle))

# --- Show first point before and after to build intuition ---
print("\n--- Intuition: what happens to point [1, 0] after 45° ---")
p0        = circle[0]                   # [1.0, 0.0] — on positive x-axis
p0_rotated = rotated_45[0]
print(f"  Before: {p0.round(4)}  (angle=  0°,  on x-axis)")
print(f"  After : {p0_rotated.round(4)}  (angle= 45°,  halfway to y-axis)")
print(f"  Norm preserved: {np.linalg.norm(p0):.4f} → {np.linalg.norm(p0_rotated):.4f}")

Unit circle points (shape (50, 2))
First 5 points:
 [[1.     0.    ]
 [0.9921 0.1253]
 [0.9686 0.2487]
 [0.9298 0.3681]
 [0.8763 0.4818]]
All norms before rotation: True

Rotated 45° points (shape (50, 2))
First 5 points:
 [[0.7071 0.7071]
 [0.6129 0.7902]
 [0.509  0.8607]
 [0.3971 0.9178]
 [0.279  0.9603]]

Norms after 45° rotation (first 10): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
All norms ≈ 1.0? True

--- Rotation matrix R (45°) ---
[[ 0.707107 -0.707107]
 [ 0.707107  0.707107]]
det(R) = 1.0   (must be 1.0 — no scaling)
R.T @ R = I? True   (R is orthogonal)

--- Rotating point [1, 0] by common angles ---
    0° → [1. 0.]
   45° → [0.7071 0.7071]
   90° → [0. 1.]
  180° → [-1.  0.]
  270° → [-0. -1.]
  360° → [ 1. -0.]

Rotate 360° returns to original? True
45° twice == 90° once? True
Rotate +73° then -73° = original? True

--- Intuition: what happens to point [1, 0] after 45° ---
  Before: [1. 0.]  (angle=  0°,  on x-axis)
  After : [0.7071 0.7071]  (angle= 45°,  halfway to y-axis)
  Norm

### How it works

- The **2D rotation matrix** R rotates any vector anticlockwise by angle θ. It is built from cos and sin of the angle and has shape `(2, 2)`.
- `points @ R.T` rotates all n points at once — shape `(n,2) @ (2,2) = (n,2)`. The transpose is needed because points are row vectors; alternatively `(R @ points.T).T` gives the same result.
- A rotation matrix has two key properties:
  - `det(R) = 1` — it rotates but never scales (preserves area)
  - `R.T @ R = I` — it is orthogonal (its inverse is just its transpose)
- Because det = 1 and R is orthogonal, rotation **preserves all vector norms** — every point on the unit circle stays on the unit circle after rotation, which is the core verification here.
- Two rotations compose perfectly: rotating by α then by β equals rotating by α+β — confirmed by the "45° twice = 90° once" check.

**Key takeaway:** The 2D rotation matrix is an orthogonal matrix with det = 1 — it rotates without stretching, so all norms and distances between points are perfectly preserved after transformation.

### Q25. Challenge — Least Squares Linear Regression from Scratch
Implement multiple linear regression **from scratch using NumPy linear algebra only** (no sklearn):
1. Generate synthetic data: `y = 3*x1 - 2*x2 + 1.5*x3 + noise` for 100 observations
2. Construct the design matrix `X` (with a bias column of ones)
3. Solve for coefficients using the Normal Equation: `β = (X.T @ X)^{-1} @ X.T @ y` using `np.linalg.solve` (more stable than using `inv` directly)
4. Also solve using `np.linalg.lstsq` and compare
5. Compute: R² score, residual sum of squares, and mean squared error
6. Verify recovered coefficients are close to the true `[1.5, 3, -2, 1.5]`
Print a coefficient comparison table (true vs estimated).

In [30]:
# Q25: Multiple linear regression from scratch — normal equation, lstsq, metrics

import numpy as np

np.random.seed(42)
n = 100   # observations

# --- Step 1: Generate synthetic data y = 1.5 + 3*x1 - 2*x2 + 1.5*x3 + noise ---
x1 = np.random.randn(n)
x2 = np.random.randn(n)
x3 = np.random.randn(n)

true_coeffs = np.array([1.5, 3.0, -2.0, 1.5])   # [bias, x1, x2, x3]
noise       = 0.5 * np.random.randn(n)

y = true_coeffs[0] + true_coeffs[1]*x1 + true_coeffs[2]*x2 + true_coeffs[3]*x3 + noise

print("--- Synthetic data ---")
print(f"True coefficients: bias={true_coeffs[0]}, x1={true_coeffs[1]}, "
      f"x2={true_coeffs[2]}, x3={true_coeffs[3]}")
print(f"y: min={y.min():.2f}  max={y.max():.2f}  mean={y.mean():.4f}")

# --- Step 2: Design matrix X with bias column of ones ---
# shape (100, 4): col0=ones (bias), col1=x1, col2=x2, col3=x3
X = np.column_stack([np.ones(n), x1, x2, x3])

print(f"\nDesign matrix X shape: {X.shape}")
print("First 5 rows:\n", X[:5].round(4))

# --- Step 3: Normal equation — β = (X.T @ X)^-1 @ X.T @ y ---
# Rewrite as linear system: (X.T @ X) @ β = X.T @ y
# np.linalg.solve is more stable than inv(X.T @ X) @ X.T @ y
XtX  = X.T @ X          # shape (4, 4) — Gram matrix
Xty  = X.T @ y          # shape (4,)

beta_normal = np.linalg.solve(XtX, Xty)   # solves XtX @ β = Xty

print("\n--- Step 3: Normal equation via np.linalg.solve ---")
print("X.T @ X (Gram matrix):\n", XtX.round(4))
print("X.T @ y:", Xty.round(4))
print("Coefficients β:", beta_normal.round(6))

# --- Step 4: lstsq solution ---
beta_lstsq, residuals_lstsq, rank, sv = np.linalg.lstsq(X, y, rcond=None)

print("\n--- Step 4: np.linalg.lstsq ---")
print("Coefficients β:", beta_lstsq.round(6))
print("Design matrix rank:", rank, " (should be 4 — full column rank)")
print("Normal == lstsq?", np.allclose(beta_normal, beta_lstsq, atol=1e-8))

# --- Step 5: Metrics ---
y_pred  = X @ beta_normal              # predicted values, shape (100,)
resid   = y - y_pred                   # residuals, shape (100,)

RSS  = np.sum(resid ** 2)              # residual sum of squares
MSE  = RSS / n                         # mean squared error
RMSE = np.sqrt(MSE)                    # root mean squared error
TSS  = np.sum((y - np.mean(y)) ** 2)  # total sum of squares
R2   = 1 - RSS / TSS                  # R² score

print("\n--- Step 5: Regression metrics ---")
print(f"RSS  (residual sum of squares) : {RSS:.6f}")
print(f"MSE  (mean squared error)      : {MSE:.6f}")
print(f"RMSE (root MSE)                : {RMSE:.6f}  ← in same units as y")
print(f"TSS  (total sum of squares)    : {TSS:.6f}")
print(f"R²   (explained variance)      : {R2:.6f}  ← 1.0 = perfect fit")

# --- Step 6: Coefficient comparison table ---
labels = ["bias (intercept)", "x1", "x2", "x3"]
print("\n--- Step 6: True vs Estimated Coefficients ---")
print(f"{'Term':<20} {'True':>10} {'Normal Eq':>12} {'lstsq':>10} {'Error':>10}")
print("-" * 65)
for name, true, est_n, est_l in zip(labels, true_coeffs, beta_normal, beta_lstsq):
    err = abs(true - est_n)
    print(f"{name:<20} {true:>10.4f} {est_n:>12.6f} {est_l:>10.6f} {err:>10.6f}")

# --- Verify residuals are centred around zero ---
print("\n--- Residual diagnostics ---")
print(f"Mean of residuals  : {resid.mean():.2e}  (should be ≈ 0)")
print(f"Std  of residuals  : {resid.std():.4f}  (should be ≈ noise std = 0.5)")
print(f"Max |residual|     : {np.max(np.abs(resid)):.4f}")

# --- Why solve > inv for normal equation ---
print("\n--- Why np.linalg.solve beats inv(X.T @ X) @ X.T @ y ---")
beta_inv = np.linalg.inv(XtX) @ Xty    # naive approach
print("Via inv  :", beta_inv.round(6))
print("Via solve:", beta_normal.round(6))
print("Same result here, but solve avoids squaring condition number of X:")
print(f"  cond(X)     = {np.linalg.cond(X):.2f}")
print(f"  cond(X.T@X) = {np.linalg.cond(XtX):.2f}  ← squared → more numerical risk")

--- Synthetic data ---
True coefficients: bias=1.5, x1=3.0, x2=-2.0, x3=1.5
y: min=-10.89  max=9.56  mean=1.2946

Design matrix X shape: (100, 4)
First 5 rows:
 [[ 1.      0.4967 -1.4154  0.3578]
 [ 1.     -0.1383 -0.4206  0.5608]
 [ 1.      0.6477 -0.3427  1.0831]
 [ 1.      1.523  -0.8023  1.0538]
 [ 1.     -0.2342 -0.1613 -1.3777]]

--- Step 3: Normal equation via np.linalg.solve ---
X.T @ X (Gram matrix):
 [[100.     -10.3847   2.2305   6.4896]
 [-10.3847  82.7306 -11.9289  17.9304]
 [  2.2305 -11.9289  90.0887  -3.6053]
 [  6.4896  17.9304  -3.6053 116.8124]]
X.T @ y: [ 129.4616  276.048  -218.6422  246.2891]
Coefficients β: [ 1.543766  2.911338 -2.019122  1.51345 ]

--- Step 4: np.linalg.lstsq ---
Coefficients β: [ 1.543766  2.911338 -2.019122  1.51345 ]
Design matrix rank: 4  (should be 4 — full column rank)
Normal == lstsq? True

--- Step 5: Regression metrics ---
RSS  (residual sum of squares) : 18.731739
MSE  (mean squared error)      : 0.187317
RMSE (root MSE)               

### How it works

- **Design matrix X** prepends a column of ones so the bias term β₀ is learned as a regular coefficient alongside β₁, β₂, β₃ — no special-casing needed.
- **Normal equation** `(X.T @ X) β = X.T @ y` is the closed-form solution that minimises the sum of squared residuals. Instead of computing `inv(X.T @ X)` directly, we pass it to `np.linalg.solve` — both give the same answer but `solve` is more numerically stable.
- **Why solve beats inv:** forming `X.T @ X` squares the condition number of X. If `cond(X) = c`, then `cond(X.T @ X) = c²`. Numerical errors grow with the condition number, so `solve` (which uses LU internally) is always safer than explicit inversion.
- **R² score** = `1 − RSS/TSS` measures the fraction of variance explained: 1.0 is a perfect fit, 0.0 means the model is no better than predicting the mean, negative means worse than the mean.
- **Residual mean ≈ 0** is a built-in property of OLS with a bias term — a quick sanity check that the fit is correct.
- `np.linalg.lstsq` also uses SVD internally, making it even more robust than the normal equation for ill-conditioned X — both methods converge to the same coefficients here because X is well-conditioned.

**Key takeaway:** Linear regression is just a linear system `Xβ = y` solved in the least-squares sense — `np.linalg.solve(X.T @ X, X.T @ y)` gives the normal equation solution, with `lstsq` as the numerically safer alternative for ill-conditioned design matrices.